In [1]:
# =============================================================================
# CELL 1 — Install Dependencies
# =============================================================================
!pip install mlflow fastapi uvicorn evidently scikit-learn xgboost lightgbm \
             pandas numpy scipy plotly seaborn matplotlib joblib pydantic \
             httpx pytest --quiet

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: C:\Users\abhir\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
# =============================================================================
# CELL 2 — Global Configuration & Setup
# =============================================================================
import os, sys, json, warnings, logging, time, hashlib, shutil, subprocess
from pathlib import Path
from datetime import datetime, timedelta
from copy import deepcopy
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
try:
    get_ipython()
    matplotlib.use('inline')
except NameError:
    matplotlib.use('Agg')

SEED = 42
np.random.seed(SEED)

# ── Enterprise Color Palette ───────────────────────────────────────────────
PAL = {
    'navy':    '#0A1628',
    'deep':    '#0D1F3C',
    'blue':    '#1A3A6B',
    'accent':  '#00D4FF',
    'gold':    '#FFB800',
    'success': '#00C896',
    'danger':  '#FF4757',
    'warning': '#FFA502',
    'purple':  '#7B2FBE',
    'neutral': '#8892A4',
    'teal':    '#2ECC71',
    'bg':      '#F0F4F8',
    'card':    '#FFFFFF',
    'text':    '#1A2332',
}

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR  = Path(r'C:\Users\abhir\OneDrive\Desktop\iitg')
MLOPS_DIR = BASE_DIR / 'mlops_platform'

SUBDIRS = [
    'pipelines', 'feature_store', 'model_registry', 'deployment',
    'monitoring', 'drift_detection', 'governance', 'dashboards',
    'apis', 'ci_cd', 'docker', 'logs', 'docs', 'tests',
    'retraining', 'inference', 'experiments', 'artifacts',
]
for d in SUBDIRS:
    (MLOPS_DIR / d).mkdir(parents=True, exist_ok=True)

# ── Logging ────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[
        logging.FileHandler(MLOPS_DIR / 'logs' / 'module13.log', mode='w', encoding='utf-8'),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger('MLOps')

def section(title):
    bar = '═' * 70
    print(f'\n{bar}\n  {title}\n{bar}')

def savefig(name, folder=None, dpi=150):
    folder = folder or MLOPS_DIR / 'dashboards'
    path   = Path(folder) / name
    plt.savefig(path, dpi=dpi, bbox_inches='tight', facecolor=PAL['bg'])
    plt.close()
    log.info('  Figure saved: %s', name)
    return path

sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams.update({
    'figure.facecolor': PAL['bg'],
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.titleweight': 'bold',
    'axes.titlesize':   12,
})

log.info('Module 13 — Enterprise MLOps Platform initialised.')
print('✅  Module 13 configuration loaded. All mlops_platform/ directories ready.')

13:52:36 | INFO     | Module 13 — Enterprise MLOps Platform initialised.
✅  Module 13 configuration loaded. All mlops_platform/ directories ready.


In [3]:
# =============================================================================
# CELL 3 — STEP 1: Enterprise MLOps Strategy Framework
# =============================================================================
section('STEP 1 — ENTERPRISE MLOPS STRATEGY FRAMEWORK')

MLOPS_STRATEGY = """
╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 13 — ENTERPRISE MLOPS STRATEGY                              ║
║  Digital Lending Portfolio Optimization                              ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  WHAT IS MLOPS IN FINTECH?                                           ║
║  MLOps is the discipline of automating, governing, and monitoring    ║
║  the entire machine learning lifecycle in production — from data     ║
║  ingestion through model deployment, monitoring and retraining.      ║
║                                                                      ║
║  WHY MLOPS MATTERS IN LENDING AI:                                    ║
║  1. Regulatory Accountability — every decision must be traceable     ║
║  2. Model Drift — borrower behavior, macro conditions change         ║
║  3. Fairness — protected groups must be monitored continuously       ║
║  4. Speed — underwriting latency directly affects conversion rates   ║
║  5. Scale — millions of decisions require automated infrastructure   ║
║  6. Fraud Adaptation — fraud tactics evolve; models must keep pace  ║
║                                                                      ║
║  MLOPS MATURITY LEVELS:                                              ║
║  Level 0 → Manual ML (notebooks, ad-hoc deployment)                 ║
║  Level 1 → ML Pipeline Automation (automated training)              ║
║  Level 2 → CI/CD for ML (automated testing + deployment)            ║
║  Level 3 → Enterprise MLOps (governance, monitoring, observability)  ║
║  → THIS MODULE TARGETS LEVEL 2–3                                     ║
║                                                                      ║
║  KEY PLATFORM COMPONENTS:                                            ║
║  ┌─────────────────────────────────────────────────────────────┐    ║
║  │  Feature Store → Training → Registry → Serving → Monitoring │    ║
║  │        ↑                                        ↓           │    ║
║  │  Data Ingestion ←────── Retraining Trigger ──────────────── │    ║
║  └─────────────────────────────────────────────────────────────┘    ║
║                                                                      ║
║  GOVERNANCE REQUIREMENTS (RBI/NBFC Compliance):                      ║
║  • Every model decision must have an audit trail                     ║
║  • Explainability required for all credit decisions                  ║
║  • Fairness monitoring across demographic groups                     ║
║  • Model approval workflow before production deployment              ║
║  • Rollback capability within 1 hour of model degradation           ║
║                                                                      ║
║  RELIABILITY OBJECTIVES (SLOs):                                      ║
║  • Underwriting API p99 latency < 200ms                             ║
║  • Fraud scoring latency < 50ms                                      ║
║  • API uptime > 99.9%                                               ║
║  • Drift alert within 1 hour of PSI > 0.20                         ║
║  • Retraining pipeline < 4 hours end-to-end                         ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(MLOPS_STRATEGY)

MLOPS_CONFIG = {
    'platform_name': 'FinLend MLOps Platform',
    'version': '1.0.0',
    'module': 'Module 13 — Enterprise MLOps',
    'models': {
        'underwriting': {'name': 'PD_Model', 'framework': 'LightGBM', 'slo_latency_ms': 200},
        'pricing':      {'name': 'Pricing_Model', 'framework': 'XGBoost', 'slo_latency_ms': 100},
        'fraud':        {'name': 'Fraud_Model', 'framework': 'Ensemble', 'slo_latency_ms': 50},
        'ews':          {'name': 'EWS_Model', 'framework': 'RandomForest', 'slo_latency_ms': 150},
    },
    'drift_thresholds': {'psi_monitor': 0.10, 'psi_alert': 0.20, 'psi_retrain': 0.25},
    'governance': {
        'approval_required': True,
        'fairness_monitoring': True,
        'explainability_required': True,
        'audit_trail': True,
    },
}
with open(MLOPS_DIR / 'docs' / 'mlops_config.json', 'w') as f:
    json.dump(MLOPS_CONFIG, f, indent=2)

print('  ✅  MLOps strategy framework saved.')
log.info('MLOps strategy framework complete.')


══════════════════════════════════════════════════════════════════════
  STEP 1 — ENTERPRISE MLOPS STRATEGY FRAMEWORK
══════════════════════════════════════════════════════════════════════

╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 13 — ENTERPRISE MLOPS STRATEGY                              ║
║  Digital Lending Portfolio Optimization                              ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  WHAT IS MLOPS IN FINTECH?                                           ║
║  MLOps is the discipline of automating, governing, and monitoring    ║
║  the entire machine learning lifecycle in production — from data     ║
║  ingestion through model deployment, monitoring and retraining.      ║
║                                                                      ║
║  WHY MLOPS MATTERS IN LENDING AI:                                    ║
║  1. R

In [4]:
# =============================================================================
# CELL 4 — STEP 2: ML Pipeline Architecture Diagram
# =============================================================================
section('STEP 2 — ML PIPELINE ARCHITECTURE')

fig, ax = plt.subplots(figsize=(22, 14))
fig.patch.set_facecolor(PAL['navy'])
ax.set_facecolor(PAL['navy'])
ax.set_xlim(0, 22); ax.set_ylim(0, 14)
ax.axis('off')

ax.text(11, 13.4, 'ENTERPRISE ML PIPELINE ARCHITECTURE',
        ha='center', fontsize=17, fontweight='bold',
        color=PAL['accent'], fontfamily='monospace')
ax.text(11, 12.9, 'Digital Lending AI Platform — Module 13 | End-to-End Production ML Flow',
        ha='center', fontsize=10, color=PAL['neutral'])

# Pipeline stages
stages = [
    ('DATA\nINGESTION', 1.1, 8.5, PAL['blue'],   PAL['accent']),
    ('DATA\nVALIDATION', 4.1, 8.5, PAL['blue'],  PAL['success']),
    ('FEATURE\nENGINEERING', 7.1, 8.5, PAL['blue'], PAL['gold']),
    ('MODEL\nTRAINING', 10.1, 8.5, PAL['blue'],  PAL['warning']),
    ('EVALUATION\n& TESTING', 13.1, 8.5, PAL['blue'], PAL['purple']),
    ('MODEL\nREGISTRY', 16.1, 8.5, PAL['blue'],  PAL['teal']),
    ('PRODUCTION\nDEPLOYMENT', 19.1, 8.5, PAL['blue'], PAL['danger']),
]

for label, x, y, bg, border in stages:
    rect = FancyBboxPatch((x - 1.2, y - 0.85), 2.4, 1.7,
        boxstyle='round,pad=0.15', facecolor=bg, edgecolor=border, linewidth=2.5)
    ax.add_patch(rect)
    ax.text(x, y + 0.05, label, ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')

# Arrows between stages
for i in range(len(stages) - 1):
    x1 = stages[i][1] + 1.2
    x2 = stages[i+1][1] - 1.2
    y_mid = stages[i][2]
    ax.annotate('', xy=(x2, y_mid), xytext=(x1, y_mid),
                arrowprops=dict(arrowstyle='->', color=PAL['accent'],
                                lw=2.0, connectionstyle='arc3,rad=0'))

# Monitoring loop (feedback)
ax.annotate('', xy=(1.1, 7.5), xytext=(19.1, 7.5),
            arrowprops=dict(arrowstyle='<-', color=PAL['danger'],
                            lw=2.5, linestyle='dashed',
                            connectionstyle='arc3,rad=-0.25'))
ax.text(10, 6.5, '◄─── MONITORING & DRIFT DETECTION FEEDBACK LOOP ───►',
        ha='center', fontsize=9, color=PAL['danger'], fontstyle='italic')

# Lower sub-components
components = [
    ('Feature Store\n(Offline/Online)', 2.5, 4.5, PAL['accent']),
    ('MLflow\nExperiment Tracker', 6.5, 4.5, PAL['gold']),
    ('Drift Detector\n(PSI/KS/SHAP)', 10.5, 4.5, PAL['warning']),
    ('FastAPI\nInference Layer', 14.5, 4.5, PAL['success']),
    ('Governance &\nAudit Logger', 18.5, 4.5, PAL['purple']),
]
for label, x, y, color in components:
    rect = FancyBboxPatch((x - 1.6, y - 0.75), 3.2, 1.5,
        boxstyle='round,pad=0.1', facecolor=PAL['deep'],
        edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center',
            fontsize=8, color=color, fontweight='bold')

ax.text(11, 2.8, 'ORCHESTRATION LAYER: Airflow / Prefect / Scheduled Jobs',
        ha='center', fontsize=10, color=PAL['neutral'],
        bbox=dict(facecolor=PAL['deep'], edgecolor=PAL['neutral'],
                  boxstyle='round,pad=0.3'))

ax.text(11, 1.7, 'INFRASTRUCTURE: Docker · FastAPI · SQLite/Postgres · Redis · Cloud Storage',
        ha='center', fontsize=9, color=PAL['neutral'])

# Legend
for i, (label, color) in enumerate([
    ('Pipeline Stage', PAL['accent']),
    ('Feedback Loop', PAL['danger']),
    ('Sub-component', PAL['gold']),
]):
    ax.text(1 + i * 5.5, 0.7, f'■ {label}', color=color, fontsize=9)

plt.tight_layout()
savefig('01_ml_pipeline_architecture.png', folder=MLOPS_DIR / 'dashboards')
print('  ✅  ML Pipeline Architecture diagram saved.')
log.info('ML Pipeline Architecture diagram complete.')


══════════════════════════════════════════════════════════════════════
  STEP 2 — ML PIPELINE ARCHITECTURE
══════════════════════════════════════════════════════════════════════
13:52:41 | INFO     |   Figure saved: 01_ml_pipeline_architecture.png
  ✅  ML Pipeline Architecture diagram saved.
13:52:41 | INFO     | ML Pipeline Architecture diagram complete.


In [5]:
# =============================================================================
# CELL 5 — STEP 3: Feature Store Design
# =============================================================================
section('STEP 3 — FEATURE STORE DESIGN')

FEATURE_STORE_CODE = '''
"""
Enterprise Feature Store — Digital Lending Platform
Supports offline (batch) and online (real-time) feature serving.
"""
import sqlite3, json, hashlib
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, List
import pandas as pd
import numpy as np

FEATURE_STORE_PATH = Path("feature_store.db")

# ── Feature Registry ────────────────────────────────────────────────────────
FEATURE_REGISTRY = {
    "credit_features": [
        "credit_score", "credit_history_length", "missed_payment_ratio",
        "avg_delay_days", "worst_delinquency_stage", "existing_loans",
    ],
    "financial_features": [
        "monthly_income", "debt_to_income_ratio", "financial_stress_index",
        "income_stability_score", "bank_balance_avg",
    ],
    "behavioral_features": [
        "spending_volatility_index", "digital_engagement_score",
        "app_usage_mean", "digital_activity_score",
    ],
    "fraud_features": [
        "income_inflation_ratio", "document_risk_proxy",
        "identity_consistency", "synthetic_id_probability",
    ],
    "loan_features": [
        "loan_amount", "loan_tenure_months", "loan_purpose",
        "acquisition_channel",
    ],
    "derived_features": [
        "pd_score", "risk_grade", "expected_loss",
        "fraud_risk_score", "health_score",
    ],
}

FEATURE_METADATA = {
    "credit_score":               {"type": "integer", "range": [300, 900], "importance": "HIGH", "sensitivity": "PII"},
    "missed_payment_ratio":       {"type": "float",   "range": [0, 1],     "importance": "HIGH", "sensitivity": "PII"},
    "monthly_income":             {"type": "float",   "range": [0, None],  "importance": "HIGH", "sensitivity": "PII"},
    "debt_to_income_ratio":       {"type": "float",   "range": [0, None],  "importance": "HIGH", "sensitivity": None},
    "financial_stress_index":     {"type": "float",   "range": [0, 1],     "importance": "HIGH", "sensitivity": None},
    "spending_volatility_index":  {"type": "float",   "range": [0, 1],     "importance": "MED",  "sensitivity": None},
    "digital_engagement_score":   {"type": "float",   "range": [0, 100],   "importance": "MED",  "sensitivity": None},
    "worst_delinquency_stage":    {"type": "integer", "range": [0, 4],     "importance": "HIGH", "sensitivity": "PII"},
    "income_stability_score":     {"type": "float",   "range": [0, 1],     "importance": "MED",  "sensitivity": None},
    "loan_amount":                {"type": "float",   "range": [0, None],  "importance": "MED",  "sensitivity": None},
}


class OfflineFeatureStore:
    """
    Batch feature store backed by SQLite.
    In production: replace with Hive/BigQuery/Feast.
    """
    def __init__(self, db_path: Path = FEATURE_STORE_PATH):
        self.db_path = db_path
        self._init_schema()

    def _connect(self):
        conn = sqlite3.connect(self.db_path, check_same_thread=False)
        conn.row_factory = sqlite3.Row
        return conn

    def _init_schema(self):
        conn = self._connect()
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS feature_vectors (
            entity_id    TEXT NOT NULL,
            feature_group TEXT NOT NULL,
            feature_name  TEXT NOT NULL,
            feature_value REAL,
            version       TEXT,
            created_at    TEXT,
            PRIMARY KEY (entity_id, feature_group, feature_name, version)
        );
        CREATE TABLE IF NOT EXISTS feature_versions (
            version       TEXT PRIMARY KEY,
            description   TEXT,
            created_at    TEXT,
            author        TEXT,
            is_active     INTEGER DEFAULT 1
        );
        CREATE TABLE IF NOT EXISTS feature_stats (
            feature_name  TEXT,
            version       TEXT,
            mean          REAL, std REAL, min_val REAL, max_val REAL,
            p25 REAL, p50 REAL, p75 REAL, null_rate REAL,
            computed_at   TEXT,
            PRIMARY KEY (feature_name, version)
        );
        """)
        conn.commit()
        conn.close()

    def write_features(self, df: pd.DataFrame, feature_group: str,
                       entity_col: str = 'loan_id',
                       version: str = 'v1') -> int:
        """Write a feature group to the offline store."""
        conn = self._connect()
        records = []
        ts = datetime.now().isoformat()
        feature_cols = [c for c in df.columns if c != entity_col]
        for _, row in df.iterrows():
            for feat in feature_cols:
                val = row[feat]
                records.append((
                    str(row[entity_col]), feature_group, feat,
                    float(val) if pd.notna(val) and isinstance(val, (int, float)) else None,
                    version, ts
                ))
        conn.executemany("""
        INSERT OR REPLACE INTO feature_vectors
        (entity_id, feature_group, feature_name, feature_value, version, created_at)
        VALUES (?,?,?,?,?,?)
        """, records)
        conn.commit()
        conn.close()
        return len(records)

    def read_features(self, entity_ids: List[str],
                      feature_group: str, version: str = 'v1') -> pd.DataFrame:
        """Retrieve features for a list of entities."""
        conn = self._connect()
        placeholders = ','.join('?' * len(entity_ids))
        rows = conn.execute(f"""
        SELECT entity_id, feature_name, feature_value
        FROM feature_vectors
        WHERE entity_id IN ({placeholders})
          AND feature_group = ?
          AND version = ?
        """, (*entity_ids, feature_group, version)).fetchall()
        conn.close()
        if not rows:
            return pd.DataFrame()
        records = [dict(r) for r in rows]
        df = pd.DataFrame(records).pivot(
            index='entity_id', columns='feature_name', values='feature_value'
        ).reset_index()
        return df

    def compute_stats(self, df: pd.DataFrame, version: str = 'v1') -> pd.DataFrame:
        """Compute and store feature statistics for monitoring."""
        stats_records = []
        ts = datetime.now().isoformat()
        for col in df.select_dtypes(include=[np.number]).columns:
            s = df[col].dropna()
            if len(s) == 0:
                continue
            stats_records.append({
                'feature_name': col,
                'version': version,
                'mean':  float(s.mean()),
                'std':   float(s.std()),
                'min_val': float(s.min()),
                'max_val': float(s.max()),
                'p25': float(s.quantile(0.25)),
                'p50': float(s.quantile(0.50)),
                'p75': float(s.quantile(0.75)),
                'null_rate': float(df[col].isna().mean()),
                'computed_at': ts,
            })
        stats_df = pd.DataFrame(stats_records)
        conn = self._connect()
        for rec in stats_records:
            conn.execute("""
            INSERT OR REPLACE INTO feature_stats
            (feature_name, version, mean, std, min_val, max_val,
             p25, p50, p75, null_rate, computed_at)
            VALUES (?,?,?,?,?,?,?,?,?,?,?)
            """, (
                rec['feature_name'], rec['version'],
                rec['mean'], rec['std'], rec['min_val'], rec['max_val'],
                rec['p25'], rec['p50'], rec['p75'],
                rec['null_rate'], rec['computed_at'],
            ))
        conn.commit()
        conn.close()
        return stats_df


class OnlineFeatureStore:
    """
    In-memory online feature cache (simulates Redis/DynamoDB in production).
    Provides sub-millisecond feature retrieval for real-time inference.
    """
    def __init__(self):
        self._cache: Dict[str, dict] = {}
        self._ttl: Dict[str, float] = {}
        self.ttl_seconds = 3600

    def set(self, entity_id: str, features: dict):
        self._cache[entity_id] = features
        self._ttl[entity_id] = time.time() + self.ttl_seconds

    def get(self, entity_id: str) -> Optional[dict]:
        if entity_id in self._cache:
            if time.time() < self._ttl.get(entity_id, 0):
                return self._cache[entity_id]
            else:
                del self._cache[entity_id]
        return None

    def bulk_set(self, entity_df: pd.DataFrame, entity_col: str = 'loan_id'):
        for _, row in entity_df.iterrows():
            self.set(str(row[entity_col]), row.to_dict())
        return len(entity_df)

    def stats(self) -> dict:
        return {'cached_entities': len(self._cache), 'ttl_seconds': self.ttl_seconds}
'''

feature_store_path = MLOPS_DIR / 'feature_store' / 'feature_store.py'
with open(feature_store_path, 'w', encoding='utf-8') as f:
    f.write(FEATURE_STORE_CODE)

print(f'  ✅  Feature Store module saved: {feature_store_path}')
log.info('Feature Store implementation complete.')


══════════════════════════════════════════════════════════════════════
  STEP 3 — FEATURE STORE DESIGN
══════════════════════════════════════════════════════════════════════
  ✅  Feature Store module saved: C:\Users\abhir\OneDrive\Desktop\iitg\mlops_platform\feature_store\feature_store.py
13:52:42 | INFO     | Feature Store implementation complete.


In [6]:
# =============================================================================
# CELL 6 — STEP 4 & 5: Model Registry + Training Pipeline
# =============================================================================
section('STEP 4 & 5 — MODEL REGISTRY + AUTOMATED TRAINING PIPELINE')

MODEL_REGISTRY_CODE = '''
"""
Model Registry & Automated Training Pipeline
Uses MLflow for experiment tracking and model versioning.
"""
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
from mlflow.tracking import MlflowClient
import json, os, hashlib, time
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    f1_score, precision_score, recall_score, confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False


# ── Model Governance Metadata ────────────────────────────────────────────────
MODEL_CARDS = {
    'PD_Model': {
        'purpose': 'Predict probability of default for loan applicants',
        'features': 8,
        'training_data': 'lending_features/master_feature_table.csv',
        'fairness_groups': ['gender', 'employment_type', 'city'],
        'performance_thresholds': {'min_auc': 0.75, 'max_brier': 0.20},
        'retraining_triggers': ['psi > 0.20', 'auc_drop > 0.05', 'monthly'],
        'regulatory_requirement': 'RBI Master Direction on Credit',
        'explainability': 'SHAP values per decision',
        'approver': 'CRO',
    },
    'Fraud_Model': {
        'purpose': 'Detect fraudulent loan applications and synthetic identities',
        'features': 12,
        'training_data': 'fraud_detection/data/fraud_labeled.csv',
        'fairness_groups': ['employment_type'],
        'performance_thresholds': {'min_auc': 0.85, 'min_precision_red': 0.70},
        'retraining_triggers': ['fraud_rate_change > 20%', 'psi > 0.15', 'weekly'],
        'regulatory_requirement': 'PMLA Anti-Money Laundering',
        'explainability': 'Rule-based + SHAP',
        'approver': 'Fraud Head',
    },
    'EWS_Model': {
        'purpose': 'Early warning detection for loan delinquency',
        'features': 10,
        'training_data': 'early_warning/data/ews_labeled.csv',
        'fairness_groups': ['employment_type'],
        'performance_thresholds': {'min_auc': 0.72, 'min_recall': 0.65},
        'retraining_triggers': ['psi > 0.20', 'recall_drop > 0.08', 'monthly'],
        'regulatory_requirement': 'RBI Prudential Norms on NPA',
        'explainability': 'SHAP + counterfactuals',
        'approver': 'CRO',
    },
}


# ── MLflow Registry Manager ──────────────────────────────────────────────────
class ModelRegistryManager:
    def __init__(self, tracking_uri: str = './mlruns',
                 registry_uri: Optional[str] = None):
        mlflow.set_tracking_uri(tracking_uri)
        if registry_uri:
            mlflow.set_registry_uri(registry_uri)
        self.client = MlflowClient()

    def get_or_create_experiment(self, name: str, tags: dict = None) -> str:
        exp = mlflow.get_experiment_by_name(name)
        if exp is None:
            exp_id = mlflow.create_experiment(name, tags=tags or {})
        else:
            exp_id = exp.experiment_id
        return exp_id

    def register_model(self, run_id: str, model_name: str,
                       metrics: dict, tags: dict = None) -> str:
        """Register a trained model in the MLflow Model Registry."""
        model_uri = f'runs:/{run_id}/model'
        try:
            result = mlflow.register_model(model_uri, model_name)
            version = result.version
            # Add governance tags
            self.client.set_model_version_tag(
                model_name, version, 'governance_status', 'PENDING_REVIEW'
            )
            self.client.set_model_version_tag(
                model_name, version, 'registered_at', datetime.now().isoformat()
            )
            for k, v in (tags or {}).items():
                self.client.set_model_version_tag(model_name, version, k, str(v))
            return version
        except Exception as e:
            return f'REGISTRY_ERROR: {e}'

    def promote_to_staging(self, model_name: str, version: str):
        self.client.transition_model_version_stage(
            model_name, version, 'Staging',
            archive_existing_versions=False
        )
        self.client.set_model_version_tag(
            model_name, version, 'governance_status', 'STAGING'
        )

    def promote_to_production(self, model_name: str, version: str,
                               approver: str = 'CRO'):
        self.client.transition_model_version_stage(
            model_name, version, 'Production',
            archive_existing_versions=True
        )
        self.client.set_model_version_tag(
            model_name, version, 'governance_status', 'APPROVED'
        )
        self.client.set_model_version_tag(
            model_name, version, 'approved_by', approver
        )
        self.client.set_model_version_tag(
            model_name, version, 'approved_at', datetime.now().isoformat()
        )

    def rollback(self, model_name: str, target_version: str):
        """Emergency rollback to a specified version."""
        self.client.transition_model_version_stage(
            model_name, target_version, 'Production',
            archive_existing_versions=True
        )
        self.client.set_model_version_tag(
            model_name, target_version, 'governance_status', 'ROLLBACK'
        )
        self.client.set_model_version_tag(
            model_name, target_version, 'rollback_at', datetime.now().isoformat()
        )

    def get_production_model(self, model_name: str):
        try:
            versions = self.client.get_latest_versions(
                model_name, stages=['Production']
            )
            return versions[0] if versions else None
        except Exception:
            return None


# ── Training Pipeline ────────────────────────────────────────────────────────
class LendingModelTrainer:
    """
    Automated training pipeline for all lending models.
    Supports experiment tracking, validation gates, and governance logging.
    """

    # Rule-based feature weights for fallback PD model
    RULE_WEIGHTS = {
        'missed_payment_ratio':     0.22,
        'worst_delinquency_stage':  0.20,
        'credit_score_norm':        0.12,
        'financial_stress_index':   0.15,
        'avg_delay_norm':           0.10,
        'debt_to_income_norm':      0.08,
        'spending_volatility_index':0.08,
        'history_norm':             0.05,
    }

    def __init__(self, registry_manager: ModelRegistryManager):
        self.registry = registry_manager

    def prepare_pd_features(self, df: pd.DataFrame) -> tuple:
        """Prepare features for PD model training."""
        feature_cols = [
            'credit_score', 'missed_payment_ratio',
            'worst_delinquency_stage', 'financial_stress_index',
            'debt_to_income_ratio', 'spending_volatility_index',
            'income_stability_score', 'pd_score',
        ]
        available = [c for c in feature_cols if c in df.columns]
        X = df[available].fillna(df[available].median())
        y = df['default_flag'].fillna(0).astype(int) if 'default_flag' in df.columns \
            else (df.get('pd_score', pd.Series(np.random.beta(2, 8, len(df)))) > 0.3).astype(int)
        return X, y

    def train_pd_model(self, df: pd.DataFrame,
                       experiment_name: str = 'PD_Model_Training') -> dict:
        """Train and register a PD model."""
        exp_id = self.registry.get_or_create_experiment(experiment_name)
        mlflow.set_experiment(experiment_name)

        X, y = self.prepare_pd_features(df)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.20, random_state=42, stratify=y
        )

        # Choose best available algorithm
        if HAS_LGB:
            model = lgb.LGBMClassifier(
                n_estimators=200, learning_rate=0.05,
                num_leaves=31, min_child_samples=20,
                random_state=42, verbosity=-1,
                class_weight='balanced',
            )
        else:
            model = GradientBoostingClassifier(
                n_estimators=200, learning_rate=0.05,
                max_depth=4, random_state=42,
            )

        with mlflow.start_run(run_name=f'PD_Train_{datetime.now().strftime("%Y%m%d_%H%M%S")}') as run:
            model.fit(X_train, y_train)
            y_prob = model.predict_proba(X_test)[:, 1]
            y_pred = (y_prob > 0.30).astype(int)

            metrics = {
                'auc_roc':  float(roc_auc_score(y_test, y_prob)),
                'auc_pr':   float(average_precision_score(y_test, y_prob)),
                'brier':    float(brier_score_loss(y_test, y_prob)),
                'precision':float(precision_score(y_test, y_pred, zero_division=0)),
                'recall':   float(recall_score(y_test, y_pred, zero_division=0)),
                'f1':       float(f1_score(y_test, y_pred, zero_division=0)),
                'n_train':  len(X_train),
                'n_test':   len(X_test),
                'default_rate': float(y.mean()),
            }

            mlflow.log_metrics(metrics)
            mlflow.log_params({
                'algorithm': type(model).__name__,
                'n_features': X.shape[1],
                'feature_names': ','.join(X.columns.tolist()),
            })
            mlflow.log_dict({'model_card': MODEL_CARDS.get('PD_Model', {})}, 'model_card.json')

            if HAS_LGB:
                mlflow.lightgbm.log_model(model, 'model')
            else:
                mlflow.sklearn.log_model(model, 'model')

            run_id = run.info.run_id

        # Governance gate
        passed = (
            metrics['auc_roc'] >= MODEL_CARDS['PD_Model']['performance_thresholds']['min_auc']
            and metrics['brier'] <= MODEL_CARDS['PD_Model']['performance_thresholds']['max_brier']
        )

        return {
            'run_id':  run_id,
            'metrics': metrics,
            'model':   model,
            'features': X.columns.tolist(),
            'governance_passed': passed,
            'gate_reason': 'AUC and Brier score within thresholds' if passed
                           else f'FAILED: AUC={metrics["auc_roc"]:.3f}',
        }

    def validate_data(self, df: pd.DataFrame) -> dict:
        """Data validation gate before training."""
        checks = {}
        checks['min_rows']     = len(df) >= 1000
        checks['has_target']   = 'default_flag' in df.columns or 'pd_score' in df.columns
        checks['low_nulls']    = df.isnull().mean().max() < 0.30
        checks['no_constants'] = all(
            df[c].nunique() > 1
            for c in df.select_dtypes(include=[np.number]).columns
        )
        checks['passed'] = all(checks[k] for k in checks if k != 'passed')
        return checks
'''

registry_path = MLOPS_DIR / 'model_registry' / 'model_registry.py'
with open(registry_path, 'w', encoding='utf-8') as f:
    f.write(MODEL_REGISTRY_CODE)

print(f'  ✅  Model Registry + Training Pipeline saved: {registry_path}')
log.info('Model Registry and Training Pipeline implementation complete.')


══════════════════════════════════════════════════════════════════════
  STEP 4 & 5 — MODEL REGISTRY + AUTOMATED TRAINING PIPELINE
══════════════════════════════════════════════════════════════════════
  ✅  Model Registry + Training Pipeline saved: C:\Users\abhir\OneDrive\Desktop\iitg\mlops_platform\model_registry\model_registry.py
13:52:44 | INFO     | Model Registry and Training Pipeline implementation complete.


In [7]:
# =============================================================================
# CELL 7 — STEP 6 & 7: FastAPI Model Serving + Real-Time Inference
# =============================================================================
section('STEP 6 & 7 — FASTAPI MODEL SERVING & REAL-TIME INFERENCE')

FASTAPI_APP_CODE = '''
"""
FinLend AI — Production Model Serving API
FastAPI application exposing underwriting, pricing, fraud, and EWS models.

Run: uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4
"""
from fastapi import FastAPI, HTTPException, Depends, BackgroundTasks, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.middleware.trustedhost import TrustedHostMiddleware
from pydantic import BaseModel, Field, validator
from typing import Optional, List
import numpy as np
import time, json, logging, hashlib, os
from datetime import datetime
from pathlib import Path

# ── App Setup ────────────────────────────────────────────────────────────────
app = FastAPI(
    title="FinLend AI — Model Serving API",
    description="Production inference layer for digital lending ML models.",
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("finlend_api")

# ── Request / Response Models ────────────────────────────────────────────────
class UnderwritingRequest(BaseModel):
    applicant_id: str = Field(..., description="Unique applicant identifier")
    monthly_income: float = Field(..., gt=0, description="Monthly income in INR")
    employment_type: str = Field(..., description="Employment category")
    credit_score: int = Field(..., ge=300, le=900)
    credit_history_length: int = Field(default=36, ge=0)
    loan_amount: float = Field(..., gt=0)
    loan_tenure_months: int = Field(..., ge=6, le=84)
    missed_payment_ratio: float = Field(default=0.05, ge=0, le=1)
    avg_delay_days: float = Field(default=5.0, ge=0)
    worst_delinquency_stage: int = Field(default=0, ge=0, le=4)
    financial_stress_index: float = Field(default=0.3, ge=0, le=1)
    debt_to_income_ratio: float = Field(default=2.0, ge=0)
    income_stability_score: float = Field(default=0.7, ge=0, le=1)
    spending_volatility_index: float = Field(default=0.25, ge=0, le=1)
    digital_engagement_score: float = Field(default=60.0, ge=0, le=100)

    @validator('employment_type')
    def validate_employment(cls, v):
        valid = ['Salaried', 'Self-Employed', 'Gig Worker', 'SME Owner', 'First-Time Borrower']
        if v not in valid:
            raise ValueError(f'employment_type must be one of {valid}')
        return v


class FraudRequest(BaseModel):
    applicant_id: str
    monthly_income: float
    employment_type: str
    credit_history_length: int
    loan_amount: float
    worst_delinquency_stage: int = 0
    income_stability_score: float = 0.7


class EWSRequest(BaseModel):
    loan_id: str
    current_dpd_stage: int = Field(default=0, ge=0, le=4)
    missed_payment_ratio: float = Field(default=0.05, ge=0, le=1)
    financial_stress_index: float = Field(default=0.3, ge=0, le=1)
    income_stability_score: float = Field(default=0.7, ge=0, le=1)
    spending_volatility_index: float = Field(default=0.25, ge=0, le=1)
    bank_balance_avg: float = Field(default=50000.0)


class UnderwritingResponse(BaseModel):
    applicant_id: str
    request_id: str
    timestamp: str
    pd_score: float
    risk_grade: str
    decision: str
    confidence: float
    interest_rate: float
    approved_amount: float
    emi_amount: float
    expected_loss: float
    lgd: float
    latency_ms: float
    model_version: str


class FraudResponse(BaseModel):
    applicant_id: str
    request_id: str
    fraud_score: float
    fraud_severity: str
    alert_level: str
    recommended_action: str
    latency_ms: float
    model_version: str


class HealthResponse(BaseModel):
    status: str
    timestamp: str
    models_loaded: dict
    uptime_seconds: float
    request_count: int


# ── Inference Engine (rule-based fallback) ────────────────────────────────────
GRADE_THRESHOLDS = {
    'A': (0.00, 0.05), 'B': (0.05, 0.10),
    'C': (0.10, 0.18), 'D': (0.18, 0.30), 'E': (0.30, 1.00),
}
LGD_BY_GRADE = {'A': 0.50, 'B': 0.55, 'C': 0.60, 'D': 0.65, 'E': 0.70}
RATE_FLOORS  = {'A': 10.5, 'B': 12.5, 'C': 15.0, 'D': 18.0, 'E': 22.0}
RATE_CEILS   = {'A': 14.0, 'B': 16.5, 'C': 20.0, 'D': 24.0, 'E': 32.0}
INCOME_MEDIANS = {
    'Salaried': 45000, 'Self-Employed': 52000, 'Gig Worker': 22000,
    'SME Owner': 85000, 'First-Time Borrower': 18000,
}

START_TIME   = time.time()
REQUEST_COUNT = 0
MODEL_VERSION = 'v1.0.0-rule-based'


def compute_pd(req: UnderwritingRequest) -> float:
    cs_norm   = max(0, (900 - req.credit_score) / 600)
    mpr_norm  = min(req.missed_payment_ratio, 1.0)
    dpd_norm  = req.worst_delinquency_stage / 4.0
    dly_norm  = min(req.avg_delay_days / 90, 1.0)
    fsi_norm  = min(req.financial_stress_index, 1.0)
    dti_norm  = min(req.debt_to_income_ratio / 10, 1.0)
    svi_norm  = min(req.spending_volatility_index, 1.0)
    hist_norm = max(0, 1 - req.credit_history_length / 60)
    raw_pd = (
        0.22 * mpr_norm + 0.20 * dpd_norm + 0.12 * cs_norm
        + 0.15 * fsi_norm + 0.10 * dly_norm + 0.08 * dti_norm
        + 0.08 * svi_norm + 0.05 * hist_norm
    )
    if req.credit_history_length < 6:
        raw_pd = min(raw_pd + 0.06, 0.95)
    if req.worst_delinquency_stage >= 3:
        raw_pd = min(raw_pd + 0.15, 0.95)
    return float(np.clip(raw_pd, 0.01, 0.95))


def get_grade(pd_score: float) -> str:
    for g, (lo, hi) in GRADE_THRESHOLDS.items():
        if lo <= pd_score < hi:
            return g
    return 'E'


def compute_emi(principal: float, rate_pct: float, months: int) -> float:
    mr = rate_pct / 100 / 12
    if mr == 0 or months == 0:
        return principal / max(months, 1)
    return principal * mr * (1 + mr)**months / ((1 + mr)**months - 1)


def generate_request_id(applicant_id: str) -> str:
    seed = f'{applicant_id}{time.time()}'
    return 'REQ' + hashlib.md5(seed.encode()).hexdigest()[:10].upper()


# ── Logging utilities ────────────────────────────────────────────────────────
PREDICTION_LOG = []

def log_prediction(request_id: str, model: str, inputs: dict,
                   outputs: dict, latency: float):
    PREDICTION_LOG.append({
        'request_id': request_id, 'model': model,
        'timestamp': datetime.now().isoformat(),
        'inputs': inputs, 'outputs': outputs, 'latency_ms': latency,
    })
    if len(PREDICTION_LOG) > 10000:
        PREDICTION_LOG.pop(0)


# ── API Routes ───────────────────────────────────────────────────────────────
@app.get('/', summary='Root')
async def root():
    return {
        'platform': 'FinLend AI Model Serving API',
        'version': '1.0.0',
        'docs': '/docs',
        'health': '/health',
        'status': 'operational',
    }


@app.get('/health', response_model=HealthResponse, summary='Health check')
async def health_check():
    global REQUEST_COUNT
    return HealthResponse(
        status='healthy',
        timestamp=datetime.now().isoformat(),
        models_loaded={
            'underwriting': MODEL_VERSION,
            'fraud': MODEL_VERSION,
            'ews': MODEL_VERSION,
        },
        uptime_seconds=round(time.time() - START_TIME, 1),
        request_count=REQUEST_COUNT,
    )


@app.post('/v1/underwriting/score', response_model=UnderwritingResponse,
          summary='Real-time loan underwriting score')
async def underwriting_score(req: UnderwritingRequest,
                              background_tasks: BackgroundTasks):
    global REQUEST_COUNT
    REQUEST_COUNT += 1
    t0 = time.time()
    request_id = generate_request_id(req.applicant_id)

    try:
        pd_score  = compute_pd(req)
        grade     = get_grade(pd_score)
        lgd       = LGD_BY_GRADE[grade]
        tenure_yr = req.loan_tenure_months / 12
        cum_pd    = 1 - (1 - pd_score) ** tenure_yr
        ecl       = cum_pd * lgd * req.loan_amount

        base_rate  = 8.5 + 2.0 + 2.0
        rate = float(np.clip(
            base_rate + pd_score * 0.55 * 100,
            RATE_FLOORS[grade], RATE_CEILS[grade]
        ))

        # Decision logic
        if pd_score > 0.55 or req.debt_to_income_ratio > 7:
            decision, confidence = 'DECLINE', 0.92
        elif grade == 'E' or pd_score > 0.30:
            decision, confidence = 'MANUAL_REVIEW', 0.70
        elif req.credit_history_length < 6:
            decision, confidence = 'MANUAL_REVIEW', 0.60
        else:
            decision = 'APPROVE'
            confidence = float(np.clip(1 - pd_score * 2, 0.55, 0.99))

        grade_caps = {'A': 1.0, 'B': 0.90, 'C': 0.80, 'D': 0.65, 'E': 0.45}
        max_emi    = req.monthly_income * 0.40
        mr_m       = rate / 100 / 12
        n_m        = req.loan_tenure_months
        max_afford = (max_emi * ((1 + mr_m)**n_m - 1) / (mr_m * (1 + mr_m)**n_m)
                      if mr_m > 0 else max_emi * n_m)
        approved   = min(req.loan_amount, max_afford * grade_caps[grade])
        emi        = compute_emi(approved, rate, n_m)
        latency    = round((time.time() - t0) * 1000, 2)

        response = UnderwritingResponse(
            applicant_id=req.applicant_id,
            request_id=request_id,
            timestamp=datetime.now().isoformat(),
            pd_score=round(pd_score, 4),
            risk_grade=grade,
            decision=decision,
            confidence=round(confidence, 3),
            interest_rate=round(rate, 2),
            approved_amount=round(approved, 2),
            emi_amount=round(emi, 2),
            expected_loss=round(ecl, 2),
            lgd=lgd,
            latency_ms=latency,
            model_version=MODEL_VERSION,
        )

        background_tasks.add_task(
            log_prediction, request_id, 'underwriting',
            req.dict(), response.dict(), latency
        )
        log.info('UW score %s | PD=%.3f | %s | %.1fms',
                 request_id, pd_score, decision, latency)
        return response

    except Exception as e:
        log.error('Scoring error %s: %s', request_id, e)
        raise HTTPException(status_code=500, detail=str(e))


@app.post('/v1/fraud/score', response_model=FraudResponse,
          summary='Real-time fraud risk score')
async def fraud_score(req: FraudRequest, background_tasks: BackgroundTasks):
    global REQUEST_COUNT
    REQUEST_COUNT += 1
    t0 = time.time()
    request_id = generate_request_id(req.applicant_id)

    exp_income = INCOME_MEDIANS.get(req.employment_type, 40000)
    income_ratio = req.monthly_income / max(exp_income, 1)
    inflation_score = min(max(income_ratio - 1.0, 0) / 2.0, 1.0)

    doc_risk = float(
        (req.credit_history_length < 6) * 0.30
        + (req.worst_delinquency_stage >= 3) * 0.20
        + (1 - req.income_stability_score) * 0.25
    )
    identity_risk = 1 - min(
        0.90 + (req.income_stability_score - 0.5) * 0.20
        - (req.credit_history_length < 6) * 0.20, 1.0
    )
    synthetic_id = float(
        (req.credit_history_length < 6) * 0.40
        + (req.loan_amount > req.monthly_income * 20) * 0.25
        + doc_risk * 0.20
    )
    fraud_score_val = float(np.clip(
        0.35 * inflation_score + 0.25 * doc_risk
        + 0.25 * identity_risk + 0.15 * synthetic_id, 0, 1
    ))

    if fraud_score_val >= 0.75:
        severity, alert, action = 'Red', 'CRITICAL', 'Block & investigate immediately'
    elif fraud_score_val >= 0.55:
        severity, alert, action = 'Orange', 'HIGH', 'Enhanced KYC verification required'
    elif fraud_score_val >= 0.35:
        severity, alert, action = 'Yellow', 'MEDIUM', 'Manual review recommended'
    else:
        severity, alert, action = 'Green', 'LOW', 'Proceed with standard processing'

    latency = round((time.time() - t0) * 1000, 2)
    response = FraudResponse(
        applicant_id=req.applicant_id,
        request_id=request_id,
        fraud_score=round(fraud_score_val, 4),
        fraud_severity=severity,
        alert_level=alert,
        recommended_action=action,
        latency_ms=latency,
        model_version=MODEL_VERSION,
    )
    background_tasks.add_task(
        log_prediction, request_id, 'fraud',
        req.dict(), response.dict(), latency
    )
    return response


@app.get('/v1/metrics/predictions', summary='Recent prediction metrics')
async def prediction_metrics():
    if not PREDICTION_LOG:
        return {'message': 'No predictions yet', 'count': 0}
    latencies = [p['latency_ms'] for p in PREDICTION_LOG]
    return {
        'total_requests': REQUEST_COUNT,
        'logged_predictions': len(PREDICTION_LOG),
        'latency_p50_ms': float(np.percentile(latencies, 50)),
        'latency_p95_ms': float(np.percentile(latencies, 95)),
        'latency_p99_ms': float(np.percentile(latencies, 99)),
        'models': list({p['model'] for p in PREDICTION_LOG}),
    }


@app.get('/v1/models', summary='List deployed models')
async def list_models():
    return {
        'models': [
            {'name': 'underwriting', 'version': MODEL_VERSION, 'status': 'production',
             'endpoint': '/v1/underwriting/score'},
            {'name': 'fraud',        'version': MODEL_VERSION, 'status': 'production',
             'endpoint': '/v1/fraud/score'},
            {'name': 'ews',          'version': MODEL_VERSION, 'status': 'production',
             'endpoint': '/v1/ews/score'},
        ]
    }
'''

api_path = MLOPS_DIR / 'apis' / 'main.py'
with open(api_path, 'w', encoding='utf-8') as f:
    f.write(FASTAPI_APP_CODE)

print(f'  ✅  FastAPI Model Serving API saved: {api_path}')
log.info('FastAPI Model Serving API complete.')


══════════════════════════════════════════════════════════════════════
  STEP 6 & 7 — FASTAPI MODEL SERVING & REAL-TIME INFERENCE
══════════════════════════════════════════════════════════════════════
  ✅  FastAPI Model Serving API saved: C:\Users\abhir\OneDrive\Desktop\iitg\mlops_platform\apis\main.py
13:52:45 | INFO     | FastAPI Model Serving API complete.


In [8]:
# =============================================================================
# CELL 8 — STEP 8: Drift Detection & Monitoring Engine
# =============================================================================
section('STEP 8 — DRIFT DETECTION & MONITORING ENGINE')

DRIFT_ENGINE_CODE = '''
"""
Enterprise Drift Detection Engine
Monitors data drift, concept drift, prediction drift, and feature stability.
"""
import numpy as np
import pandas as pd
from scipy import stats
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime
import json, logging

log = logging.getLogger('DriftDetector')

# ── Drift Result Data Model ────────────────────────────────────────────────
@dataclass
class DriftResult:
    feature:       str
    drift_type:    str   # data / concept / prediction / fairness
    metric:        str   # PSI / KS / chi2 / wasserstein
    score:         float
    threshold_monitor: float
    threshold_alert:   float
    threshold_retrain: float
    status:        str   # OK / MONITOR / ALERT / RETRAIN
    direction:     str   # UP / DOWN / STABLE
    timestamp:     str = field(default_factory=lambda: datetime.now().isoformat())
    recommendation: str = ''

    @property
    def needs_action(self):
        return self.status in ('ALERT', 'RETRAIN')


@dataclass
class DriftReport:
    report_id:   str
    timestamp:   str
    baseline_period: str
    current_period:  str
    n_features_drifted: int
    overall_status: str  # GREEN / YELLOW / RED
    results: List[DriftResult] = field(default_factory=list)
    escalation_required: bool = False
    retraining_required: bool = False


class DriftDetector:
    """
    Comprehensive drift detection for lending ML models.
    Implements PSI, KS-test, Chi-squared, and Wasserstein distance.
    """

    PSI_THRESHOLDS    = {'monitor': 0.10, 'alert': 0.20, 'retrain': 0.25}
    KS_THRESHOLDS     = {'monitor': 0.05, 'alert': 0.10, 'retrain': 0.15}
    PRED_THRESHOLDS   = {'monitor': 0.03, 'alert': 0.06, 'retrain': 0.10}

    def compute_psi(self, baseline: np.ndarray,
                    current: np.ndarray,
                    n_bins: int = 10) -> float:
        """
        Population Stability Index (PSI).
        PSI < 0.10: Stable
        PSI 0.10–0.20: Monitor
        PSI 0.20–0.25: Alert
        PSI > 0.25: Retrain
        """
        eps = 1e-6
        try:
            bins = np.percentile(baseline, np.linspace(0, 100, n_bins + 1))
            bins = np.unique(bins)
            if len(bins) < 3:
                return 0.0
            base_counts = np.histogram(baseline, bins=bins)[0]
            curr_counts = np.histogram(current,  bins=bins)[0]
            base_pct = np.maximum(base_counts / len(baseline), eps)
            curr_pct = np.maximum(curr_counts / len(current),  eps)
            base_pct /= base_pct.sum()
            curr_pct /= curr_pct.sum()
            psi = np.sum((curr_pct - base_pct) * np.log(curr_pct / base_pct))
            return float(np.clip(psi, 0, None))
        except Exception:
            return 0.0

    def compute_ks(self, baseline: np.ndarray,
                   current: np.ndarray) -> Tuple[float, float]:
        """Kolmogorov-Smirnov test for distributional drift."""
        try:
            ks_stat, p_value = stats.ks_2samp(baseline, current)
            return float(ks_stat), float(p_value)
        except Exception:
            return 0.0, 1.0

    def compute_wasserstein(self, baseline: np.ndarray,
                             current: np.ndarray) -> float:
        """Wasserstein (Earth Mover's) distance — sensitive to shape changes."""
        try:
            return float(stats.wasserstein_distance(baseline, current))
        except Exception:
            return 0.0

    def _get_status(self, score: float, thresholds: dict) -> str:
        if score >= thresholds['retrain']:
            return 'RETRAIN'
        elif score >= thresholds['alert']:
            return 'ALERT'
        elif score >= thresholds['monitor']:
            return 'MONITOR'
        return 'OK'

    def _get_direction(self, baseline: np.ndarray,
                       current: np.ndarray) -> str:
        if current.mean() > baseline.mean() * 1.05:
            return 'UP'
        elif current.mean() < baseline.mean() * 0.95:
            return 'DOWN'
        return 'STABLE'

    def detect_feature_drift(self, baseline_df: pd.DataFrame,
                              current_df: pd.DataFrame,
                              features: Optional[List[str]] = None) -> List[DriftResult]:
        """Run drift detection across all numeric features."""
        numeric_cols = baseline_df.select_dtypes(include=[np.number]).columns.tolist()
        target_features = features or numeric_cols
        target_features = [f for f in target_features if f in baseline_df.columns
                            and f in current_df.columns]

        results = []
        for feat in target_features:
            base_arr = baseline_df[feat].dropna().values
            curr_arr = current_df[feat].dropna().values
            if len(base_arr) < 30 or len(curr_arr) < 30:
                continue

            psi_score  = self.compute_psi(base_arr, curr_arr)
            ks_stat, _ = self.compute_ks(base_arr, curr_arr)
            status = self._get_status(psi_score, self.PSI_THRESHOLDS)
            direction = self._get_direction(base_arr, curr_arr)

            recommendation = ''
            if status == 'RETRAIN':
                recommendation = f'Feature {feat} has drifted significantly. Trigger retraining pipeline.'
            elif status == 'ALERT':
                recommendation = f'Feature {feat} showing drift. Increase monitoring frequency.'
            elif status == 'MONITOR':
                recommendation = f'Feature {feat} showing early drift signals. Watch closely.'

            results.append(DriftResult(
                feature=feat,
                drift_type='data',
                metric='PSI',
                score=round(psi_score, 4),
                threshold_monitor=self.PSI_THRESHOLDS['monitor'],
                threshold_alert=self.PSI_THRESHOLDS['alert'],
                threshold_retrain=self.PSI_THRESHOLDS['retrain'],
                status=status,
                direction=direction,
                recommendation=recommendation,
            ))
        return results

    def detect_prediction_drift(self, baseline_preds: np.ndarray,
                                 current_preds: np.ndarray) -> DriftResult:
        """Detect drift in model prediction distribution."""
        psi_score = self.compute_psi(baseline_preds, current_preds)
        ks_stat, _ = self.compute_ks(baseline_preds, current_preds)
        mean_shift = abs(current_preds.mean() - baseline_preds.mean())
        combined_score = 0.5 * psi_score + 0.3 * ks_stat + 0.2 * min(mean_shift / 0.1, 1.0)
        status = self._get_status(combined_score, self.PRED_THRESHOLDS)
        direction = self._get_direction(baseline_preds, current_preds)

        return DriftResult(
            feature='pd_score_predictions',
            drift_type='prediction',
            metric='PSI+KS+MeanShift',
            score=round(combined_score, 4),
            threshold_monitor=self.PRED_THRESHOLDS['monitor'],
            threshold_alert=self.PRED_THRESHOLDS['alert'],
            threshold_retrain=self.PRED_THRESHOLDS['retrain'],
            status=status,
            direction=direction,
            recommendation=(
                'Model predictions shifting. Investigate data pipeline and retrain.'
                if status in ('ALERT', 'RETRAIN') else
                'Prediction distribution stable.'
            ),
        )

    def detect_fairness_drift(self, baseline_df: pd.DataFrame,
                               current_df: pd.DataFrame,
                               group_col: str,
                               score_col: str = 'pd_score') -> List[DriftResult]:
        """Monitor fairness metrics across demographic groups."""
        results = []
        if group_col not in baseline_df.columns or score_col not in baseline_df.columns:
            return results
        for group in baseline_df[group_col].unique():
            base_group = baseline_df[baseline_df[group_col] == group][score_col].dropna().values
            curr_group = current_df[current_df[group_col] == group][score_col].dropna().values
            if len(base_group) < 20 or len(curr_group) < 20:
                continue
            psi = self.compute_psi(base_group, curr_group)
            status = self._get_status(psi, self.PSI_THRESHOLDS)
            results.append(DriftResult(
                feature=f'{score_col}@{group_col}={group}',
                drift_type='fairness',
                metric='PSI',
                score=round(psi, 4),
                threshold_monitor=self.PSI_THRESHOLDS['monitor'],
                threshold_alert=self.PSI_THRESHOLDS['alert'],
                threshold_retrain=self.PSI_THRESHOLDS['retrain'],
                status=status,
                direction=self._get_direction(base_group, curr_group),
                recommendation=(
                    f'Fairness drift for group {group} in {group_col}. CRO review required.'
                    if status in ('ALERT', 'RETRAIN') else ''
                ),
            ))
        return results

    def generate_report(self, all_results: List[DriftResult],
                         baseline_period: str = 'T-30d',
                         current_period: str = 'T-0d') -> DriftReport:
        """Consolidate drift results into an actionable report."""
        n_drifted = sum(1 for r in all_results if r.needs_action)
        needs_retrain = any(r.status == 'RETRAIN' for r in all_results)
        needs_escalation = any(r.status in ('ALERT', 'RETRAIN') for r in all_results)

        if needs_retrain:
            overall = 'RED'
        elif needs_escalation:
            overall = 'YELLOW'
        else:
            overall = 'GREEN'

        import hashlib, time
        report_id = 'DRF' + hashlib.md5(
            f'{time.time()}'.encode()
        ).hexdigest()[:8].upper()

        return DriftReport(
            report_id=report_id,
            timestamp=datetime.now().isoformat(),
            baseline_period=baseline_period,
            current_period=current_period,
            n_features_drifted=n_drifted,
            overall_status=overall,
            results=all_results,
            escalation_required=needs_escalation,
            retraining_required=needs_retrain,
        )
'''

drift_path = MLOPS_DIR / 'drift_detection' / 'drift_detector.py'
with open(drift_path, 'w', encoding='utf-8') as f:
    f.write(DRIFT_ENGINE_CODE)

print(f'  ✅  Drift Detection Engine saved: {drift_path}')
log.info('Drift Detection Engine complete.')


══════════════════════════════════════════════════════════════════════
  STEP 8 — DRIFT DETECTION & MONITORING ENGINE
══════════════════════════════════════════════════════════════════════
  ✅  Drift Detection Engine saved: C:\Users\abhir\OneDrive\Desktop\iitg\mlops_platform\drift_detection\drift_detector.py
13:52:47 | INFO     | Drift Detection Engine complete.


In [9]:
# =============================================================================
# CELL 9 — STEP 8 continued: Simulate Drift + Visualize
# =============================================================================
section('DRIFT SIMULATION & VISUALIZATION')

np.random.seed(SEED)
N_BASE = 5000
N_CURR = 2000

# Simulate baseline portfolio features
df_baseline = pd.DataFrame({
    'credit_score':           np.random.randint(400, 850, N_BASE),
    'pd_score':               np.random.beta(2, 8, N_BASE),
    'missed_payment_ratio':   np.random.beta(1.5, 10, N_BASE),
    'financial_stress_index': np.random.beta(2, 6, N_BASE),
    'spending_volatility':    np.random.beta(2, 6, N_BASE),
    'debt_to_income_ratio':   np.random.gamma(2, 1.2, N_BASE),
    'income_stability_score': np.random.beta(5, 2, N_BASE),
    'loan_amount':            np.random.uniform(30000, 500000, N_BASE),
    'employment_type':        np.random.choice(
        ['Salaried', 'Self-Employed', 'Gig Worker', 'SME Owner', 'First-Time Borrower'], N_BASE
    ),
})

# Simulate current period with drift in some features
df_current = pd.DataFrame({
    'credit_score':           np.random.randint(350, 780, N_CURR),      # DRIFT DOWN
    'pd_score':               np.random.beta(3, 7, N_CURR),             # DRIFT UP
    'missed_payment_ratio':   np.random.beta(2.5, 8, N_CURR),           # DRIFT UP
    'financial_stress_index': np.random.beta(3, 5, N_CURR),             # DRIFT UP
    'spending_volatility':    np.random.beta(2, 6, N_CURR),             # STABLE
    'debt_to_income_ratio':   np.random.gamma(2.8, 1.2, N_CURR),        # DRIFT UP
    'income_stability_score': np.random.beta(4, 2.5, N_CURR),           # SLIGHT DRIFT
    'loan_amount':            np.random.uniform(30000, 500000, N_CURR),  # STABLE
    'employment_type':        np.random.choice(
        ['Salaried', 'Self-Employed', 'Gig Worker', 'SME Owner', 'First-Time Borrower'],
        N_CURR, p=[0.35, 0.20, 0.25, 0.12, 0.08]  # shifted toward Gig
    ),
})

# Compute PSI manually for visualization
def compute_psi(baseline, current, n_bins=10):
    eps = 1e-6
    bins = np.percentile(baseline, np.linspace(0, 100, n_bins + 1))
    bins = np.unique(bins)
    if len(bins) < 3:
        return 0.0
    bp = np.maximum(np.histogram(baseline, bins=bins)[0] / len(baseline), eps)
    cp = np.maximum(np.histogram(current,  bins=bins)[0] / len(current),  eps)
    bp /= bp.sum(); cp /= cp.sum()
    return float(np.sum((cp - bp) * np.log(cp / bp + eps)))

def compute_ks(baseline, current):
    return float(stats.ks_2samp(baseline, current)[0])

numeric_feats = [c for c in df_baseline.columns if df_baseline[c].dtype != object]
drift_results = []
for feat in numeric_feats:
    psi = compute_psi(df_baseline[feat].values, df_current[feat].values)
    ks  = compute_ks(df_baseline[feat].values, df_current[feat].values)
    if psi >= 0.25: status, color = 'RETRAIN', PAL['danger']
    elif psi >= 0.20: status, color = 'ALERT', PAL['warning']
    elif psi >= 0.10: status, color = 'MONITOR', PAL['gold']
    else: status, color = 'OK', PAL['success']
    drift_results.append({'feature': feat, 'psi': psi, 'ks': ks,
                           'status': status, 'color': color})

drift_df = pd.DataFrame(drift_results).sort_values('psi', ascending=False)
drift_df.to_csv(MLOPS_DIR / 'drift_detection' / 'drift_report.csv', index=False)

# ── DRIFT DASHBOARD ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
fig.patch.set_facecolor(PAL['bg'])
fig.suptitle('DRIFT DETECTION & MONITORING DASHBOARD\nDigital Lending AI Platform — Module 13',
             fontsize=15, fontweight='bold', color=PAL['text'], y=0.98)

# P1: PSI by Feature
ax = axes[0, 0]; ax.set_facecolor('white')
bars = ax.barh(drift_df['feature'], drift_df['psi'],
               color=drift_df['color'], edgecolor='white', lw=1.5)
ax.axvline(0.10, color=PAL['gold'],    lw=2, linestyle='--', label='Monitor (0.10)')
ax.axvline(0.20, color=PAL['warning'], lw=2, linestyle='--', label='Alert (0.20)')
ax.axvline(0.25, color=PAL['danger'],  lw=2, linestyle='--', label='Retrain (0.25)')
ax.set_title('Feature PSI — Population Stability Index', fontweight='bold')
ax.set_xlabel('PSI Score')
ax.legend(fontsize=8)
for i, row in enumerate(drift_df.itertuples()):
    ax.text(row.psi + 0.002, i, f'{row.psi:.3f}', va='center', fontsize=8,
            color=row.color, fontweight='bold')

# P2: KS Statistic
ax = axes[0, 1]; ax.set_facecolor('white')
ks_colors = [PAL['danger'] if v > 0.10 else PAL['warning'] if v > 0.05 else PAL['success']
              for v in drift_df['ks']]
ax.barh(drift_df['feature'], drift_df['ks'], color=ks_colors, edgecolor='white', lw=1.5)
ax.axvline(0.05, color=PAL['warning'], lw=2, linestyle='--', label='Alert (0.05)')
ax.axvline(0.10, color=PAL['danger'],  lw=2, linestyle='--', label='Critical (0.10)')
ax.set_title('KS Test Statistic', fontweight='bold')
ax.set_xlabel('KS Statistic')
ax.legend(fontsize=8)

# P3: Drift Status Summary
ax = axes[0, 2]; ax.set_facecolor('white')
status_counts = drift_df['status'].value_counts()
order = ['RETRAIN', 'ALERT', 'MONITOR', 'OK']
s_colors = [PAL['danger'], PAL['warning'], PAL['gold'], PAL['success']]
sv = [status_counts.get(s, 0) for s in order]
wedges, texts, atexts = ax.pie(
    sv, labels=order, colors=s_colors,
    autopct=lambda p: f'{p:.0f}%' if p > 0 else '',
    startangle=90, pctdistance=0.7,
    textprops={'fontsize': 9}
)
for at in atexts: at.set_fontsize(8)
ax.set_title('Drift Status Distribution', fontweight='bold')

# P4: PD Score Drift (baseline vs current)
ax = axes[1, 0]; ax.set_facecolor('white')
ax.hist(df_baseline['pd_score'], bins=40, alpha=0.65, density=True,
        color=PAL['success'], label=f'Baseline (n={N_BASE:,})', edgecolor='white')
ax.hist(df_current['pd_score'],  bins=40, alpha=0.65, density=True,
        color=PAL['danger'],  label=f'Current (n={N_CURR:,})',  edgecolor='white')
ax.axvline(df_baseline['pd_score'].mean(), color=PAL['success'], lw=2, linestyle='--',
            label=f'Base mean={df_baseline["pd_score"].mean():.3f}')
ax.axvline(df_current['pd_score'].mean(),  color=PAL['danger'],  lw=2, linestyle='--',
            label=f'Curr mean={df_current["pd_score"].mean():.3f}')
ax.set_title('Prediction Drift: PD Score Distribution', fontweight='bold')
ax.set_xlabel('PD Score')
ax.set_ylabel('Density')
ax.legend(fontsize=8)

# P5: Credit Score Drift
ax = axes[1, 1]; ax.set_facecolor('white')
ax.hist(df_baseline['credit_score'], bins=40, alpha=0.65, density=True,
        color=PAL['blue'],   label='Baseline', edgecolor='white')
ax.hist(df_current['credit_score'],  bins=40, alpha=0.65, density=True,
        color=PAL['warning'], label='Current (drift)', edgecolor='white')
ax.axvline(df_baseline['credit_score'].mean(), color=PAL['blue'],    lw=2, linestyle='--')
ax.axvline(df_current['credit_score'].mean(),  color=PAL['warning'], lw=2, linestyle='--')
ax.set_title('Feature Drift: Credit Score', fontweight='bold')
ax.set_xlabel('Credit Score')
ax.set_ylabel('Density')
ax.legend(fontsize=8)
psi_cs = compute_psi(df_baseline['credit_score'].values, df_current['credit_score'].values)
ax.text(0.98, 0.95, f'PSI = {psi_cs:.3f}\n{"⚠ ALERT" if psi_cs > 0.20 else "OK"}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        color=PAL['danger'] if psi_cs > 0.20 else PAL['success'], fontweight='bold')

# P6: Drift Timeline
ax = axes[1, 2]; ax.set_facecolor('white')
np.random.seed(SEED)
n_weeks = 16
psi_timeline = np.clip(
    0.03 + np.cumsum(np.random.normal(0.008, 0.015, n_weeks)), 0, 0.40
)
weeks = [f'W{i+1}' for i in range(n_weeks)]
alert_mask  = psi_timeline > 0.20
monitor_mask = (psi_timeline > 0.10) & ~alert_mask
ax.fill_between(range(n_weeks), psi_timeline, alpha=0.25, color=PAL['warning'])
ax.plot(range(n_weeks), psi_timeline, color=PAL['warning'], lw=2.5,
        marker='o', markersize=5, label='PSI (PD Score)')
ax.axhline(0.10, color=PAL['gold'],    lw=2, linestyle='--', label='Monitor (0.10)')
ax.axhline(0.20, color=PAL['warning'], lw=2, linestyle='--', label='Alert (0.20)')
ax.axhline(0.25, color=PAL['danger'],  lw=2, linestyle='--', label='Retrain (0.25)')
ax.scatter([i for i, m in enumerate(alert_mask) if m],
            [psi_timeline[i] for i, m in enumerate(alert_mask) if m],
            color=PAL['danger'], s=80, zorder=5, label='Alert')
ax.set_xticks(range(n_weeks))
ax.set_xticklabels(weeks, rotation=45, fontsize=7)
ax.set_title('PSI Timeline — 16-Week Drift Monitor', fontweight='bold')
ax.set_ylabel('PSI Score')
ax.legend(fontsize=8)

plt.tight_layout()
savefig('02_drift_detection_dashboard.png', folder=MLOPS_DIR / 'dashboards')
print('  ✅  Drift Detection Dashboard saved.')
log.info('Drift Detection and Monitoring complete.')


══════════════════════════════════════════════════════════════════════
  DRIFT SIMULATION & VISUALIZATION
══════════════════════════════════════════════════════════════════════
13:52:50 | INFO     |   Figure saved: 02_drift_detection_dashboard.png
  ✅  Drift Detection Dashboard saved.
13:52:50 | INFO     | Drift Detection and Monitoring complete.


In [10]:
# =============================================================================
# CELL 10 — STEP 9 & 13: Governance Engine + Monitoring Dashboard
# =============================================================================
section('STEP 9 & 13 — GOVERNANCE ENGINE + MONITORING DASHBOARDS')

GOVERNANCE_ENGINE_CODE = '''
"""
AI Governance & Auditability Engine
Manages model approvals, prediction logging, fairness monitoring,
audit trails, and regulatory compliance workflows.
"""
import sqlite3, json, hashlib, time
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, List
from dataclasses import dataclass
import numpy as np
import pandas as pd

GOV_DB_PATH = Path('governance.db')

@dataclass
class GovernanceDecision:
    decision_id:    str
    model_name:     str
    model_version:  str
    applicant_id:   str
    prediction:     float
    decision:       str
    confidence:     float
    top_features:   dict
    explanation:    str
    fairness_group: str
    timestamp:      str
    latency_ms:     float
    reviewed:       bool = False


class GovernanceEngine:
    """
    Central governance registry for all model decisions.
    Implements RBI-grade audit trail and fairness monitoring.
    """

    REGULATORY_CHECKS = {
        'RBI_FPC': 'Fair Practices Code — Explanation required for all declinations',
        'RBI_KYC': 'Know Your Customer — fraud score must be logged',
        'IND_AS_109': 'Expected Credit Loss — ECL must be computed and logged',
        'PMLA': 'Anti-Money Laundering — suspicious patterns must be flagged',
        'DI_RULE': 'Disparate Impact — 80% rule monitoring required',
    }

    def __init__(self, db_path: Path = GOV_DB_PATH):
        self.db_path = db_path
        self._init_schema()

    def _connect(self):
        conn = sqlite3.connect(self.db_path, check_same_thread=False)
        conn.row_factory = sqlite3.Row
        return conn

    def _init_schema(self):
        conn = self._connect()
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS decision_audit (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            decision_id     TEXT UNIQUE,
            model_name      TEXT,
            model_version   TEXT,
            applicant_id    TEXT,
            prediction      REAL,
            decision        TEXT,
            confidence      REAL,
            top_features    TEXT,
            explanation     TEXT,
            fairness_group  TEXT,
            timestamp       TEXT,
            latency_ms      REAL,
            reviewed        INTEGER DEFAULT 0
        );
        CREATE TABLE IF NOT EXISTS model_approvals (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            model_name      TEXT,
            version         TEXT,
            action          TEXT,
            approver        TEXT,
            rationale       TEXT,
            metrics_json    TEXT,
            approved_at     TEXT,
            effective_date  TEXT
        );
        CREATE TABLE IF NOT EXISTS fairness_log (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            model_name      TEXT,
            evaluation_date TEXT,
            group_col       TEXT,
            group_value     TEXT,
            approval_rate   REAL,
            avg_pd_score    REAL,
            di_ratio        REAL,
            di_pass         INTEGER,
            notes           TEXT
        );
        CREATE TABLE IF NOT EXISTS governance_alerts (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            alert_id        TEXT,
            alert_type      TEXT,
            severity        TEXT,
            description     TEXT,
            triggered_at    TEXT,
            resolved        INTEGER DEFAULT 0,
            resolved_at     TEXT
        );
        """)
        conn.commit()
        conn.close()

    def log_decision(self, gov_decision: GovernanceDecision):
        conn = self._connect()
        conn.execute("""
        INSERT OR IGNORE INTO decision_audit
        (decision_id, model_name, model_version, applicant_id,
         prediction, decision, confidence, top_features, explanation,
         fairness_group, timestamp, latency_ms, reviewed)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            gov_decision.decision_id, gov_decision.model_name,
            gov_decision.model_version, gov_decision.applicant_id,
            gov_decision.prediction, gov_decision.decision,
            gov_decision.confidence,
            json.dumps(gov_decision.top_features),
            gov_decision.explanation, gov_decision.fairness_group,
            gov_decision.timestamp, gov_decision.latency_ms,
            int(gov_decision.reviewed),
        ))
        conn.commit()
        conn.close()

    def log_model_approval(self, model_name: str, version: str,
                            action: str, approver: str,
                            rationale: str, metrics: dict):
        conn = self._connect()
        conn.execute("""
        INSERT INTO model_approvals
        (model_name, version, action, approver, rationale, metrics_json, approved_at, effective_date)
        VALUES (?,?,?,?,?,?,?,?)
        """, (
            model_name, version, action, approver, rationale,
            json.dumps(metrics),
            datetime.now().isoformat(),
            datetime.now().isoformat(),
        ))
        conn.commit()
        conn.close()

    def compute_fairness(self, df: pd.DataFrame,
                          group_col: str,
                          score_col: str = 'pd_score',
                          decision_col: str = 'decision',
                          model_name: str = 'PD_Model') -> pd.DataFrame:
        """
        Compute disparate impact (4/5ths rule) across demographic groups.
        Groups with approval_rate < 80% of highest group flag as CONCERN.
        """
        if group_col not in df.columns:
            return pd.DataFrame()
        results = []
        if decision_col in df.columns:
            group_stats = df.groupby(group_col).apply(lambda g: pd.Series({
                'approval_rate': (g[decision_col] == 'APPROVE').mean() if decision_col in g else np.nan,
                'avg_pd_score':  g[score_col].mean() if score_col in g else np.nan,
                'count':         len(g),
            })).reset_index()
        else:
            group_stats = df.groupby(group_col).apply(lambda g: pd.Series({
                'approval_rate': (g[score_col] < 0.30).mean() if score_col in g else np.nan,
                'avg_pd_score':  g[score_col].mean() if score_col in g else np.nan,
                'count':         len(g),
            })).reset_index()

        max_rate = group_stats['approval_rate'].max()
        group_stats['di_ratio'] = group_stats['approval_rate'] / max(max_rate, 0.001)
        group_stats['di_pass']  = group_stats['di_ratio'] >= 0.80
        group_stats['model_name'] = model_name
        group_stats['evaluation_date'] = datetime.now().isoformat()

        # Persist to DB
        conn = self._connect()
        for _, row in group_stats.iterrows():
            conn.execute("""
            INSERT INTO fairness_log
            (model_name, evaluation_date, group_col, group_value,
             approval_rate, avg_pd_score, di_ratio, di_pass)
            VALUES (?,?,?,?,?,?,?,?)
            """, (
                model_name, row['evaluation_date'], group_col, str(row[group_col]),
                row['approval_rate'], row['avg_pd_score'],
                row['di_ratio'], int(row['di_pass']),
            ))
        conn.commit()
        conn.close()
        return group_stats

    def raise_alert(self, alert_type: str, severity: str, description: str):
        alert_id = 'ALT' + hashlib.md5(f'{time.time()}'.encode()).hexdigest()[:8].upper()
        conn = self._connect()
        conn.execute("""
        INSERT INTO governance_alerts (alert_id, alert_type, severity, description, triggered_at)
        VALUES (?,?,?,?,?)
        """, (alert_id, alert_type, severity, description, datetime.now().isoformat()))
        conn.commit()
        conn.close()
        return alert_id

    def get_audit_summary(self, days: int = 30) -> dict:
        conn = self._connect()
        total   = conn.execute('SELECT COUNT(*) FROM decision_audit').fetchone()[0]
        by_dec  = dict(conn.execute(
            'SELECT decision, COUNT(*) FROM decision_audit GROUP BY decision'
        ).fetchall())
        avg_lat = conn.execute(
            'SELECT AVG(latency_ms) FROM decision_audit'
        ).fetchone()[0] or 0
        alerts  = conn.execute(
            'SELECT COUNT(*) FROM governance_alerts WHERE resolved=0'
        ).fetchone()[0]
        conn.close()
        return {
            'total_decisions': total,
            'by_decision': by_dec,
            'avg_latency_ms': round(avg_lat, 2),
            'open_alerts': alerts,
        }
'''

gov_path = MLOPS_DIR / 'governance' / 'governance_engine.py'
with open(gov_path, 'w', encoding='utf-8') as f:
    f.write(GOVERNANCE_ENGINE_CODE)

print(f'  ✅  Governance Engine saved: {gov_path}')
log.info('Governance Engine complete.')


══════════════════════════════════════════════════════════════════════
  STEP 9 & 13 — GOVERNANCE ENGINE + MONITORING DASHBOARDS
══════════════════════════════════════════════════════════════════════
  ✅  Governance Engine saved: C:\Users\abhir\OneDrive\Desktop\iitg\mlops_platform\governance\governance_engine.py
13:52:52 | INFO     | Governance Engine complete.


In [11]:
# =============================================================================
# CELL 11 — STEP 14: Automated Retraining Framework
# =============================================================================
section('STEP 14 — AUTOMATED RETRAINING FRAMEWORK')

RETRAINING_CODE = '''
"""
Automated Retraining Orchestrator
Manages trigger-based, scheduled, and governance-approved retraining.
"""
import time, json, logging, hashlib
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Callable
from dataclasses import dataclass, field
from enum import Enum
import numpy as np
import pandas as pd

log = logging.getLogger('Retraining')


class TriggerType(str, Enum):
    SCHEDULED  = 'scheduled'
    DRIFT      = 'drift'
    PERFORMANCE = 'performance'
    FAIRNESS   = 'fairness'
    MANUAL     = 'manual'
    FRAUD_RATE = 'fraud_rate'


@dataclass
class RetrainingJob:
    job_id:      str
    model_name:  str
    trigger:     TriggerType
    trigger_reason: str
    requested_at: str = field(default_factory=lambda: datetime.now().isoformat())
    started_at:  Optional[str] = None
    completed_at: Optional[str] = None
    status:      str = 'QUEUED'  # QUEUED / RUNNING / PASSED / FAILED / BLOCKED
    metrics_before: dict = field(default_factory=dict)
    metrics_after:  dict = field(default_factory=dict)
    promoted:    bool = False
    approver:    Optional[str] = None
    error:       Optional[str] = None


@dataclass
class RetrainingTrigger:
    name:      str
    trigger_type: TriggerType
    condition: Callable
    priority:  int = 5   # 1=highest, 10=lowest
    cooldown_hours: int = 24   # Minimum hours between retraining


class RetrainingOrchestrator:
    """
    Manages the full automated retraining lifecycle:
    trigger detection → data validation → training → evaluation → promotion.
    """

    # Governance thresholds
    PERFORMANCE_THRESHOLDS = {
        'PD_Model':    {'min_auc': 0.75, 'max_brier': 0.20, 'max_psi': 0.20},
        'Fraud_Model': {'min_auc': 0.85, 'min_precision': 0.70, 'max_psi': 0.15},
        'EWS_Model':   {'min_auc': 0.72, 'min_recall': 0.65, 'max_psi': 0.20},
    }

    def __init__(self, log_dir: Path = Path('retraining_logs')):
        self.log_dir = log_dir
        self.log_dir.mkdir(parents=True, exist_ok=True)
        self.job_history: List[RetrainingJob] = []
        self.last_retrained: Dict[str, float] = {}
        self._setup_triggers()

    def _setup_triggers(self):
        """Define all retraining triggers."""
        self.triggers = [
            RetrainingTrigger(
                name='psi_critical_drift',
                trigger_type=TriggerType.DRIFT,
                condition=lambda ctx: ctx.get('max_psi', 0) >= 0.25,
                priority=1, cooldown_hours=6,
            ),
            RetrainingTrigger(
                name='auc_degradation',
                trigger_type=TriggerType.PERFORMANCE,
                condition=lambda ctx: ctx.get('current_auc', 1) < ctx.get('baseline_auc', 0.8) - 0.05,
                priority=2, cooldown_hours=12,
            ),
            RetrainingTrigger(
                name='fairness_violation',
                trigger_type=TriggerType.FAIRNESS,
                condition=lambda ctx: ctx.get('min_di_ratio', 1) < 0.80,
                priority=2, cooldown_hours=12,
            ),
            RetrainingTrigger(
                name='fraud_rate_spike',
                trigger_type=TriggerType.FRAUD_RATE,
                condition=lambda ctx: ctx.get('current_fraud_rate', 0) > ctx.get('baseline_fraud_rate', 0) * 1.30,
                priority=1, cooldown_hours=4,
            ),
            RetrainingTrigger(
                name='scheduled_monthly',
                trigger_type=TriggerType.SCHEDULED,
                condition=lambda ctx: ctx.get('days_since_last_retrain', 0) >= 30,
                priority=8, cooldown_hours=25 * 24,
            ),
            RetrainingTrigger(
                name='psi_alert_drift',
                trigger_type=TriggerType.DRIFT,
                condition=lambda ctx: ctx.get('max_psi', 0) >= 0.20,
                priority=3, cooldown_hours=12,
            ),
        ]

    def evaluate_triggers(self, model_name: str, context: dict) -> Optional[RetrainingTrigger]:
        """
        Evaluate all triggers. Return the highest-priority trigger that fires.
        Respects cooldown periods.
        """
        last_time = self.last_retrained.get(model_name, 0)
        fired = []
        for trigger in sorted(self.triggers, key=lambda t: t.priority):
            hours_since = (time.time() - last_time) / 3600
            if hours_since < trigger.cooldown_hours:
                continue
            try:
                if trigger.condition(context):
                    fired.append(trigger)
            except Exception:
                pass
        return fired[0] if fired else None

    def create_job(self, model_name: str, trigger: RetrainingTrigger,
                   context: dict) -> RetrainingJob:
        job_id = 'RTJ' + hashlib.md5(f'{model_name}{time.time()}'.encode()).hexdigest()[:8].upper()
        reason = f'{trigger.trigger_type}: {trigger.name}'
        if trigger.trigger_type == TriggerType.DRIFT:
            reason += f' | PSI={context.get("max_psi", 0):.3f}'
        elif trigger.trigger_type == TriggerType.PERFORMANCE:
            reason += f' | AUC={context.get("current_auc", 0):.3f} (baseline={context.get("baseline_auc", 0):.3f})'
        job = RetrainingJob(
            job_id=job_id,
            model_name=model_name,
            trigger=trigger.trigger_type,
            trigger_reason=reason,
        )
        self.job_history.append(job)
        return job

    def run_retraining_simulation(self, job: RetrainingJob,
                                    df: pd.DataFrame) -> RetrainingJob:
        """
        Simulate the retraining pipeline (replace with actual training call in production).
        """
        job.status = 'RUNNING'
        job.started_at = datetime.now().isoformat()
        log.info('Starting retraining job %s for %s', job.job_id, job.model_name)

        # Simulate pipeline steps
        pipeline_steps = [
            'data_validation',
            'feature_engineering',
            'model_training',
            'evaluation',
            'governance_gate',
        ]
        step_results = {}
        for step in pipeline_steps:
            time.sleep(0.01)  # Simulate work
            step_results[step] = 'PASSED'

        # Simulate metrics
        thresholds = self.PERFORMANCE_THRESHOLDS.get(job.model_name, {'min_auc': 0.70})
        before_auc = np.random.uniform(0.72, 0.78)
        after_auc  = np.random.uniform(0.77, 0.85)

        job.metrics_before = {'auc_roc': round(before_auc, 4), 'brier': 0.18}
        job.metrics_after  = {'auc_roc': round(after_auc,  4), 'brier': 0.14}

        governance_passed = after_auc >= thresholds.get('min_auc', 0.70)
        job.status       = 'PASSED' if governance_passed else 'FAILED'
        job.completed_at = datetime.now().isoformat()

        self.last_retrained[job.model_name] = time.time()
        log.info('Retraining job %s completed: %s | AUC %.3f→%.3f',
                  job.job_id, job.status, before_auc, after_auc)

        # Save job record
        job_record = {
            'job_id': job.job_id, 'model_name': job.model_name,
            'trigger': job.trigger, 'reason': job.trigger_reason,
            'status': job.status, 'metrics_before': job.metrics_before,
            'metrics_after': job.metrics_after,
            'requested_at': job.requested_at, 'completed_at': job.completed_at,
        }
        log_file = self.log_dir / f'{job.job_id}.json'
        with open(log_file, 'w') as f:
            json.dump(job_record, f, indent=2)

        return job

    def get_job_summary(self) -> pd.DataFrame:
        if not self.job_history:
            return pd.DataFrame()
        records = [{
            'job_id': j.job_id, 'model': j.model_name,
            'trigger': j.trigger, 'status': j.status,
            'auc_before': j.metrics_before.get('auc_roc', np.nan),
            'auc_after':  j.metrics_after.get('auc_roc', np.nan),
            'requested_at': j.requested_at,
        } for j in self.job_history]
        return pd.DataFrame(records)
'''

retrain_path = MLOPS_DIR / 'retraining' / 'retraining_orchestrator.py'
with open(retrain_path, 'w', encoding='utf-8') as f:
    f.write(RETRAINING_CODE)

print(f'  ✅  Automated Retraining Framework saved: {retrain_path}')
log.info('Automated Retraining Framework complete.')


══════════════════════════════════════════════════════════════════════
  STEP 14 — AUTOMATED RETRAINING FRAMEWORK
══════════════════════════════════════════════════════════════════════
  ✅  Automated Retraining Framework saved: C:\Users\abhir\OneDrive\Desktop\iitg\mlops_platform\retraining\retraining_orchestrator.py
13:52:54 | INFO     | Automated Retraining Framework complete.


In [12]:
# =============================================================================
# CELL 12 — STEP 10: CI/CD Pipeline Configuration
# =============================================================================
section('STEP 10 — CI/CD PIPELINE CONFIGURATION')

GITHUB_ACTIONS_YAML = '''name: FinLend AI — MLOps CI/CD Pipeline

on:
  push:
    branches: [main, staging]
    paths:
      - 'mlops_platform/**'
      - 'models/**'
      - 'apis/**'
  pull_request:
    branches: [main]
  schedule:
    - cron: '0 2 * * *'  # Daily at 02:00 UTC — scheduled drift check
  workflow_dispatch:
    inputs:
      model_name:
        description: 'Model to retrain (PD_Model / Fraud_Model / EWS_Model)'
        required: false
      force_retrain:
        description: 'Force retraining regardless of drift'
        type: boolean
        default: false

env:
  PYTHON_VERSION: '3.10'
  MLFLOW_TRACKING_URI: ${{ secrets.MLFLOW_TRACKING_URI }}
  API_BASE_URL: ${{ secrets.API_BASE_URL }}

jobs:

  # ──────────────────────────────────────────────────────────────────────────
  # Job 1: Lint & Code Quality
  # ──────────────────────────────────────────────────────────────────────────
  lint:
    name: Code Quality & Linting
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v4
        with:
          python-version: ${{ env.PYTHON_VERSION }}
      - name: Install linting tools
        run: pip install flake8 black isort
      - name: Run flake8
        run: flake8 mlops_platform/ --max-line-length=120 --ignore=E501,W503
      - name: Check formatting
        run: black --check mlops_platform/

  # ──────────────────────────────────────────────────────────────────────────
  # Job 2: Unit & Integration Tests
  # ──────────────────────────────────────────────────────────────────────────
  test:
    name: Unit & Integration Tests
    runs-on: ubuntu-latest
    needs: lint
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v4
        with:
          python-version: ${{ env.PYTHON_VERSION }}
      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install pytest pytest-cov httpx
      - name: Run unit tests
        run: pytest mlops_platform/tests/ -v --cov=mlops_platform --cov-report=xml
      - name: API smoke test
        run: |
          uvicorn mlops_platform.apis.main:app --host 0.0.0.0 --port 8000 &
          sleep 5
          python mlops_platform/tests/test_api_smoke.py
      - name: Upload coverage
        uses: codecov/codecov-action@v3
        with:
          file: coverage.xml

  # ──────────────────────────────────────────────────────────────────────────
  # Job 3: Drift Detection Check
  # ──────────────────────────────────────────────────────────────────────────
  drift_check:
    name: Drift Detection & Monitoring
    runs-on: ubuntu-latest
    needs: test
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v4
        with:
          python-version: ${{ env.PYTHON_VERSION }}
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Run drift detection
        run: python mlops_platform/drift_detection/run_drift_check.py
      - name: Upload drift report
        uses: actions/upload-artifact@v3
        with:
          name: drift-report
          path: mlops_platform/drift_detection/drift_report.json
      - name: Trigger retraining if needed
        if: ${{ env.RETRAIN_NEEDED == 'true' }}
        run: python mlops_platform/retraining/trigger_retrain.py

  # ──────────────────────────────────────────────────────────────────────────
  # Job 4: Model Training & Validation
  # ──────────────────────────────────────────────────────────────────────────
  train_validate:
    name: Model Training & Governance Validation
    runs-on: ubuntu-latest
    needs: drift_check
    if: |
      github.event.inputs.force_retrain == 'true' ||
      env.RETRAIN_NEEDED == 'true'
    steps:
      - uses: actions/checkout@v4
      - name: Train model
        run: python mlops_platform/pipelines/train_pipeline.py
      - name: Governance validation gate
        run: python mlops_platform/governance/governance_gate.py
      - name: Upload model artifacts
        uses: actions/upload-artifact@v3
        with:
          name: model-artifacts
          path: mlops_platform/artifacts/

  # ──────────────────────────────────────────────────────────────────────────
  # Job 5: Build & Push Docker Image
  # ──────────────────────────────────────────────────────────────────────────
  build_docker:
    name: Build & Push Docker Image
    runs-on: ubuntu-latest
    needs: train_validate
    steps:
      - uses: actions/checkout@v4
      - name: Log in to Container Registry
        uses: docker/login-action@v2
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}
      - name: Build and push
        uses: docker/build-push-action@v4
        with:
          context: ./mlops_platform
          file: ./mlops_platform/docker/Dockerfile
          push: true
          tags: ghcr.io/${{ github.repository }}/finlend-api:${{ github.sha }}

  # ──────────────────────────────────────────────────────────────────────────
  # Job 6: Deploy to Staging
  # ──────────────────────────────────────────────────────────────────────────
  deploy_staging:
    name: Deploy to Staging
    runs-on: ubuntu-latest
    needs: build_docker
    environment: staging
    steps:
      - name: Deploy to staging
        run: |
          echo "Deploying to staging environment"
          # kubectl apply -f k8s/staging/ or docker-compose up
      - name: Run staging smoke tests
        run: python mlops_platform/tests/test_staging_smoke.py
      - name: Notify Slack
        uses: 8398a7/action-slack@v3
        with:
          status: ${{ job.status }}
          text: 'FinLend AI deployed to staging — ${{ github.sha }}'
        env:
          SLACK_WEBHOOK_URL: ${{ secrets.SLACK_WEBHOOK }}

  # ──────────────────────────────────────────────────────────────────────────
  # Job 7: Production Deployment (manual approval required)
  # ──────────────────────────────────────────────────────────────────────────
  deploy_production:
    name: Deploy to Production (CRO Approval)
    runs-on: ubuntu-latest
    needs: deploy_staging
    environment:
      name: production
      url: https://api.finlend.ai
    steps:
      - name: Deploy to production
        run: |
          echo "Deploying to production — approved by ${{ github.actor }}"
      - name: Update model registry status
        run: python mlops_platform/model_registry/promote_to_prod.py
      - name: Enable canary rollout (10% traffic)
        run: |
          echo "Canary rollout: 10% traffic → new model"
      - name: Notify CRO
        run: echo "Production deployment complete"
'''

# Save GitHub Actions workflow
ci_cd_dir = MLOPS_DIR / 'ci_cd'
with open(ci_cd_dir / 'mlops_pipeline.yml', 'w', encoding='utf-8') as f:
    f.write(GITHUB_ACTIONS_YAML)

print(f'  ✅  GitHub Actions CI/CD pipeline saved: {ci_cd_dir / "mlops_pipeline.yml"}')
log.info('CI/CD Pipeline configuration complete.')


══════════════════════════════════════════════════════════════════════
  STEP 10 — CI/CD PIPELINE CONFIGURATION
══════════════════════════════════════════════════════════════════════
  ✅  GitHub Actions CI/CD pipeline saved: C:\Users\abhir\OneDrive\Desktop\iitg\mlops_platform\ci_cd\mlops_pipeline.yml
13:52:55 | INFO     | CI/CD Pipeline configuration complete.


In [14]:
# =============================================================================
# CELL 13 — STEP 16: Docker & Deployment Configuration
# =============================================================================
section('STEP 16 — DOCKER & DEPLOYMENT CONFIGURATION')

DOCKERFILE = '''# FinLend AI — Model Serving API
# Base: Python 3.10-slim for minimal footprint
FROM python:3.10-slim

LABEL maintainer="FinLend AI Platform"
LABEL description="Production model serving API for digital lending"
LABEL version="1.0.0"

# Security: run as non-root
RUN groupadd -r finlend && useradd -r -g finlend finlend

# System dependencies
RUN apt-get update && apt-get install -y \\
    libgomp1 curl git \\
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# Install Python dependencies first (layer caching)
COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip && \\
    pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY apis/ ./apis/
COPY model_registry/ ./model_registry/
COPY drift_detection/ ./drift_detection/
COPY governance/ ./governance/
COPY feature_store/ ./feature_store/
COPY artifacts/ ./artifacts/

# Switch to non-root user
RUN chown -R finlend:finlend /app
USER finlend

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=40s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

EXPOSE 8000

# Production server: 4 workers, uvicorn
CMD ["uvicorn", "apis.main:app",
     "--host", "0.0.0.0",
     "--port", "8000",
     "--workers", "4",
     "--log-level", "info",
     "--access-log"]
'''

DOCKER_COMPOSE = '''version: '3.9'

services:
  # Model Serving API
  finlend-api:
    build:
      context: .
      dockerfile: docker/Dockerfile
    image: finlend-api:latest
    container_name: finlend-api
    ports:
      - "8000:8000"
    environment:
      - MLFLOW_TRACKING_URI=http://mlflow:5000
      - ENVIRONMENT=production
      - LOG_LEVEL=info
    volumes:
      - ./artifacts:/app/artifacts:ro
      - ./logs:/app/logs
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
    restart: unless-stopped
    depends_on:
      - mlflow

  # MLflow Tracking Server
  mlflow:
    image: python:3.10-slim
    container_name: mlflow-server
    command: >
      bash -c "pip install mlflow && mlflow server
        --backend-store-uri sqlite:////mlflow/mlflow.db
        --default-artifact-root /mlflow/artifacts
        --host 0.0.0.0 --port 5000"
    ports:
      - "5000:5000"
    volumes:
      - mlflow_data:/mlflow
    restart: unless-stopped

  # Streamlit Dashboard
  streamlit:
    image: python:3.10-slim
    container_name: finlend-dashboard
    command: >
      bash -c "pip install streamlit pandas plotly && streamlit run app.py
        --server.port 8501 --server.headless true"
    ports:
      - "8501:8501"
    volumes:
      - ./:/app
    working_dir: /app
    restart: unless-stopped

volumes:
  mlflow_data:

networks:
  default:
    name: finlend-network
'''

REQUIREMENTS_TXT = '''# FinLend AI Platform — Production Requirements
numpy>=1.24.0
pandas>=2.0.0
scikit-learn>=1.3.0
lightgbm>=4.0.0
xgboost>=2.0.0
scipy>=1.11.0
joblib>=1.3.0
mlflow>=2.8.0
fastapi>=0.104.0
uvicorn[standard]>=0.24.0
pydantic>=2.4.0
httpx>=0.25.0
evidently>=0.4.0
streamlit>=1.28.0
plotly>=5.18.0
matplotlib>=3.8.0
seaborn>=0.13.0
pytest>=7.4.0
pytest-cov>=4.1.0
python-dotenv>=1.0.0
tqdm>=4.66.0
openpyxl>=3.1.0
'''

with open(MLOPS_DIR / 'docker' / 'Dockerfile', 'w', encoding='utf-8') as f:
    f.write(DOCKERFILE)
with open(MLOPS_DIR / 'docker' / 'docker-compose.yml', 'w', encoding='utf-8') as f:
    f.write(DOCKER_COMPOSE)
with open(MLOPS_DIR / 'requirements.txt', 'w', encoding='utf-8') as f:
    f.write(REQUIREMENTS_TXT)

print('  ✅  Dockerfile, docker-compose.yml, requirements.txt saved.')
log.info('Docker & Deployment configuration complete.')


══════════════════════════════════════════════════════════════════════
  STEP 16 — DOCKER & DEPLOYMENT CONFIGURATION
══════════════════════════════════════════════════════════════════════
  ✅  Dockerfile, docker-compose.yml, requirements.txt saved.
13:53:53 | INFO     | Docker & Deployment configuration complete.


In [15]:
# =============================================================================
# CELL 14 — STEP 15 & 17: Security + Ops Playbook + Monitoring Dashboard
# =============================================================================
section('STEP 15 & 17 — SECURITY, OPS PLAYBOOK & MONITORING DASHBOARD')

# ── SIMULATED INFERENCE MONITORING DATA ─────────────────────────────────────
np.random.seed(SEED)
n_hours = 168  # 1 week of hourly data

timestamps = pd.date_range('2024-12-01', periods=n_hours, freq='h')
monitoring_df = pd.DataFrame({
    'timestamp':       timestamps,
    'requests_total':  np.random.poisson(450, n_hours),
    'latency_p50_ms':  np.clip(45 + np.random.normal(0, 8, n_hours), 20, 300),
    'latency_p99_ms':  np.clip(180 + np.random.normal(0, 30, n_hours), 50, 800),
    'error_rate_pct':  np.clip(np.random.exponential(0.3, n_hours), 0, 5),
    'pd_score_mean':   np.clip(0.18 + np.cumsum(np.random.normal(0, 0.001, n_hours)), 0.05, 0.60),
    'fraud_alerts':    np.random.poisson(8, n_hours) + (np.arange(n_hours) > 120) * 5,
    'approvals':       np.random.poisson(310, n_hours),
    'declines':        np.random.poisson(85, n_hours),
    'manual_reviews':  np.random.poisson(55, n_hours),
    'model_uptime':    np.clip(99.9 - np.random.exponential(0.05, n_hours), 98.0, 100.0),
})

monitoring_df['hour'] = monitoring_df['timestamp'].dt.hour
monitoring_df['day']  = monitoring_df['timestamp'].dt.day_of_week

# ── MONITORING DASHBOARD ─────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(24, 18))
fig.patch.set_facecolor(PAL['bg'])
fig.suptitle(
    'MLOPS PRODUCTION MONITORING DASHBOARD\n'
    'Digital Lending AI Platform — Module 13 | 7-Day Operational View',
    fontsize=15, fontweight='bold', color=PAL['text'], y=0.98
)

# P1: Request Volume
ax = axes[0, 0]; ax.set_facecolor('white')
ax.fill_between(monitoring_df['timestamp'], monitoring_df['requests_total'],
                alpha=0.3, color=PAL['accent'])
ax.plot(monitoring_df['timestamp'], monitoring_df['requests_total'],
        color=PAL['accent'], lw=1.5)
ax.set_title('API Request Volume (per hour)', fontweight='bold')
ax.set_ylabel('Requests/hr')
ax.tick_params(axis='x', rotation=30, labelsize=7)

# P2: Latency P50/P99
ax = axes[0, 1]; ax.set_facecolor('white')
ax.plot(monitoring_df['timestamp'], monitoring_df['latency_p50_ms'],
        color=PAL['success'], lw=2, label='P50 latency')
ax.plot(monitoring_df['timestamp'], monitoring_df['latency_p99_ms'],
        color=PAL['warning'], lw=2, label='P99 latency')
ax.axhline(200, color=PAL['danger'], lw=1.5, linestyle='--', label='SLO (200ms p99)')
ax.set_title('API Latency: P50 & P99 (ms)', fontweight='bold')
ax.set_ylabel('Latency (ms)')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=30, labelsize=7)

# P3: Error Rate
ax = axes[0, 2]; ax.set_facecolor('white')
err_colors = [PAL['danger'] if v > 1.0 else PAL['warning'] if v > 0.5 else PAL['success']
               for v in monitoring_df['error_rate_pct']]
ax.bar(monitoring_df['timestamp'], monitoring_df['error_rate_pct'],
       color=err_colors, width=pd.Timedelta(hours=0.8))
ax.axhline(1.0, color=PAL['danger'], lw=2, linestyle='--', label='Error SLO (1%)')
ax.set_title('API Error Rate (%)', fontweight='bold')
ax.set_ylabel('Error Rate (%)')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=30, labelsize=7)

# P4: PD Score Drift Over Time
ax = axes[1, 0]; ax.set_facecolor('white')
ax.fill_between(monitoring_df['timestamp'], monitoring_df['pd_score_mean'],
                alpha=0.3, color=PAL['warning'])
ax.plot(monitoring_df['timestamp'], monitoring_df['pd_score_mean'],
        color=PAL['warning'], lw=2)
ax.axhline(0.15, color=PAL['danger'], lw=2, linestyle='--', label='Risk appetite (15%)')
ax.axhline(monitoring_df['pd_score_mean'].iloc[0], color=PAL['success'],
            lw=1.5, linestyle='--', label='Baseline')
ax.set_title('Live PD Score Mean (Drift Indicator)', fontweight='bold')
ax.set_ylabel('Avg PD Score')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=30, labelsize=7)

# P5: Fraud Alerts
ax = axes[1, 1]; ax.set_facecolor('white')
spike = monitoring_df['fraud_alerts'] > monitoring_df['fraud_alerts'].quantile(0.90)
normal_color = [PAL['danger'] if s else PAL['warning'] for s in spike]
ax.bar(monitoring_df['timestamp'], monitoring_df['fraud_alerts'],
       color=normal_color, width=pd.Timedelta(hours=0.8))
ax.axhline(monitoring_df['fraud_alerts'].mean() * 1.5, color=PAL['danger'],
            lw=2, linestyle='--', label='Alert threshold')
ax.set_title('Fraud Alert Volume', fontweight='bold')
ax.set_ylabel('# Alerts')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=30, labelsize=7)

# P6: Decision Distribution Over Time
ax = axes[1, 2]; ax.set_facecolor('white')
total_d = (monitoring_df['approvals'] + monitoring_df['declines']
            + monitoring_df['manual_reviews'])
ax.stackplot(
    monitoring_df['timestamp'],
    monitoring_df['approvals'] / total_d.clip(1) * 100,
    monitoring_df['manual_reviews'] / total_d.clip(1) * 100,
    monitoring_df['declines'] / total_d.clip(1) * 100,
    labels=['Approve', 'Manual Review', 'Decline'],
    colors=[PAL['success'], PAL['warning'], PAL['danger']],
    alpha=0.8,
)
ax.set_title('Decision Mix Over Time (%)', fontweight='bold')
ax.set_ylabel('Share (%)')
ax.set_ylim(0, 100)
ax.legend(fontsize=8, loc='upper left')
ax.tick_params(axis='x', rotation=30, labelsize=7)

# P7: Model Uptime
ax = axes[2, 0]; ax.set_facecolor('white')
up_colors = [PAL['success'] if v > 99.5 else PAL['warning'] if v > 99.0 else PAL['danger']
              for v in monitoring_df['model_uptime']]
ax.fill_between(monitoring_df['timestamp'], monitoring_df['model_uptime'],
                98.0, alpha=0.4, color=PAL['success'])
ax.plot(monitoring_df['timestamp'], monitoring_df['model_uptime'],
        color=PAL['success'], lw=1.5)
ax.axhline(99.9, color=PAL['danger'], lw=2, linestyle='--', label='SLO (99.9%)')
ax.set_title('Model Serving Uptime (%)', fontweight='bold')
ax.set_ylabel('Uptime (%)')
ax.set_ylim(97.5, 100.5)
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=30, labelsize=7)

# P8: Hourly Heatmap (requests by hour of day)
ax = axes[2, 1]; ax.set_facecolor('white')
hourly_avg = monitoring_df.groupby('hour')['requests_total'].mean()
ax.bar(hourly_avg.index, hourly_avg.values, color=PAL['blue'], edgecolor='white', lw=1)
ax.set_title('Avg Request Volume by Hour of Day', fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Requests/hr')
ax.set_xticks(range(24))

# P9: SLO Summary Scorecard
ax = axes[2, 2]
ax.set_facecolor('white')
ax.axis('off')
ax.set_title('SLO Compliance Scorecard', fontweight='bold', fontsize=12)

slo_checks = [
    ('API Uptime > 99.9%',    monitoring_df['model_uptime'].mean() >= 99.9,
     f'{monitoring_df["model_uptime"].mean():.2f}%'),
    ('P99 Latency < 200ms',   monitoring_df['latency_p99_ms'].quantile(0.95) < 200,
     f'{monitoring_df["latency_p99_ms"].quantile(0.95):.0f}ms'),
    ('Error Rate < 1%',       monitoring_df['error_rate_pct'].mean() < 1.0,
     f'{monitoring_df["error_rate_pct"].mean():.2f}%'),
    ('PD Drift < 15%',        monitoring_df['pd_score_mean'].max() < 0.15,
     f'{monitoring_df["pd_score_mean"].max():.3f}'),
    ('Fraud SLO < 3x baseline', monitoring_df['fraud_alerts'].max() < 30,
     f'{monitoring_df["fraud_alerts"].max():.0f} max alerts'),
]

for i, (label, ok, actual) in enumerate(slo_checks):
    y = 0.88 - i * 0.17
    icon  = '✅' if ok else '❌'
    color = PAL['success'] if ok else PAL['danger']
    ax.text(0.05, y, f'{icon}  {label}', va='center', fontsize=9,
            color=color, transform=ax.transAxes, fontweight='bold')
    ax.text(0.80, y, actual, va='center', fontsize=9,
            color=PAL['text'], transform=ax.transAxes)

plt.tight_layout()
savefig('03_mlops_monitoring_dashboard.png', folder=MLOPS_DIR / 'dashboards')
print('  ✅  MLOps Monitoring Dashboard saved.')
log.info('Monitoring Dashboard complete.')


══════════════════════════════════════════════════════════════════════
  STEP 15 & 17 — SECURITY, OPS PLAYBOOK & MONITORING DASHBOARD
══════════════════════════════════════════════════════════════════════
13:53:57 | INFO     |   Figure saved: 03_mlops_monitoring_dashboard.png
  ✅  MLOps Monitoring Dashboard saved.
13:53:57 | INFO     | Monitoring Dashboard complete.


In [16]:
# =============================================================================
# CELL 15 — STEP 18: Advanced Visual Analytics
# =============================================================================
section('STEP 18 — ADVANCED VISUAL ANALYTICS')

# ── 1. RETRAINING TIMELINE GANTT ─────────────────────────────────────────────
np.random.seed(SEED)

retrain_jobs = [
    {'model': 'PD_Model',    'trigger': 'Monthly Schedule', 'start': 0,  'duration': 3.2, 'status': 'PASSED'},
    {'model': 'Fraud_Model', 'trigger': 'PSI Alert',        'start': 5,  'duration': 2.1, 'status': 'PASSED'},
    {'model': 'EWS_Model',   'trigger': 'AUC Degradation',  'start': 12, 'duration': 2.8, 'status': 'PASSED'},
    {'model': 'PD_Model',    'trigger': 'PSI Critical',     'start': 20, 'duration': 3.5, 'status': 'PASSED'},
    {'model': 'Fraud_Model', 'trigger': 'Monthly Schedule', 'start': 30, 'duration': 2.2, 'status': 'FAILED'},
    {'model': 'Fraud_Model', 'trigger': 'Retry',            'start': 33, 'duration': 2.5, 'status': 'PASSED'},
    {'model': 'PD_Model',    'trigger': 'Fairness Drift',   'start': 41, 'duration': 3.8, 'status': 'PASSED'},
    {'model': 'EWS_Model',   'trigger': 'Monthly Schedule', 'start': 45, 'duration': 2.9, 'status': 'PASSED'},
]

fig, axes = plt.subplots(2, 2, figsize=(22, 14))
fig.patch.set_facecolor(PAL['bg'])
fig.suptitle('ADVANCED MLOPS ANALYTICS\nRetraining Timeline · Model Lineage · Pipeline Orchestration',
             fontsize=14, fontweight='bold', color=PAL['text'], y=0.98)

# Gantt: Retraining Timeline
ax = axes[0, 0]; ax.set_facecolor('white')
model_y = {'PD_Model': 2, 'Fraud_Model': 1, 'EWS_Model': 0}
color_map = {'PASSED': PAL['success'], 'FAILED': PAL['danger']}

for job in retrain_jobs:
    y = model_y[job['model']]
    color = color_map[job['status']]
    ax.barh(y, job['duration'], left=job['start'], height=0.6,
             color=color, alpha=0.85, edgecolor='white', lw=1.5)
    ax.text(job['start'] + job['duration'] / 2, y, job['trigger'],
             ha='center', va='center', fontsize=6.5, color='white', fontweight='bold')

ax.set_yticks(list(model_y.values()))
ax.set_yticklabels(list(model_y.keys()), fontsize=10)
ax.set_xlabel('Days')
ax.set_title('Retraining Timeline (48-Day Window)', fontweight='bold')
passed_patch = plt.Rectangle((0,0), 1, 1, fc=PAL['success'], label='PASSED')
failed_patch = plt.Rectangle((0,0), 1, 1, fc=PAL['danger'],  label='FAILED')
ax.legend(handles=[passed_patch, failed_patch], fontsize=9)

# AUC Performance Trend
ax = axes[0, 1]; ax.set_facecolor('white')
weeks_12 = range(12)
pd_auc  = np.clip(0.82 + np.cumsum(np.random.normal(-0.004, 0.012, 12)), 0.65, 0.95)
fr_auc  = np.clip(0.91 + np.cumsum(np.random.normal(-0.003, 0.010, 12)), 0.75, 0.98)
ews_auc = np.clip(0.77 + np.cumsum(np.random.normal(-0.005, 0.011, 12)), 0.60, 0.92)

ax.plot(weeks_12, pd_auc,  color=PAL['accent'],   lw=2.5, marker='o', ms=5, label='PD Model')
ax.plot(weeks_12, fr_auc,  color=PAL['success'],  lw=2.5, marker='s', ms=5, label='Fraud Model')
ax.plot(weeks_12, ews_auc, color=PAL['warning'],  lw=2.5, marker='^', ms=5, label='EWS Model')
ax.axhline(0.75, color=PAL['danger'], lw=2, linestyle='--', label='Min AUC threshold')

# Mark retraining events
retrain_weeks = [2, 5, 8]
for rw in retrain_weeks:
    ax.axvline(rw, color=PAL['purple'], lw=1.5, linestyle=':', alpha=0.7)
ax.text(retrain_weeks[0] + 0.1, 0.65, '↑ Retrain', fontsize=7, color=PAL['purple'])

ax.set_title('Model Performance Trend (AUC-ROC)', fontweight='bold')
ax.set_xlabel('Week')
ax.set_ylabel('AUC-ROC')
ax.legend(fontsize=8)
ax.set_xticks(range(12))
ax.set_xticklabels([f'W{i+1}' for i in range(12)], fontsize=8)

# Model Lineage Graph (text-based)
ax = axes[1, 0]
ax.set_facecolor(PAL['navy'])
ax.axis('off')
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.set_title('MODEL LINEAGE & VERSION GRAPH', fontweight='bold',
              color=PAL['accent'], fontsize=10)

lineage_nodes = [
    ('Raw Data', 1.5, 8.5, PAL['neutral']),
    ('Feature Store v1', 4, 8.5, PAL['blue']),
    ('PD Model v1.0', 7, 8.5, PAL['gold']),
    ('Feature Store v2', 4, 6.0, PAL['accent']),
    ('PD Model v2.0', 7, 6.0, PAL['success']),
    ('Drift Alert', 4, 4.0, PAL['danger']),
    ('PD Model v2.1', 7, 4.0, PAL['success']),
    ('PRODUCTION', 7, 2.0, PAL['gold']),
]
for label, x, y, color in lineage_nodes:
    rect = FancyBboxPatch((x - 1.3, y - 0.4), 2.6, 0.8,
        boxstyle='round,pad=0.1', facecolor=PAL['deep'], edgecolor=color, lw=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=7,
            color=color, fontweight='bold')

edges = [
    (1.5, 8.1, 4, 8.9), (4, 8.1, 7, 8.9), (4, 8.1, 4, 6.4),
    (4, 5.6, 7, 6.4), (7, 5.6, 7, 4.4), (4, 3.6, 7, 4.4),
    (7, 3.6, 7, 2.4),
]
for x1, y1, x2, y2 in edges:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=PAL['neutral'],
                                lw=1.5))

# Fairness Monitoring Over Time
ax = axes[1, 1]; ax.set_facecolor('white')
n_months = 12
groups = ['Salaried', 'Self-Employed', 'Gig Worker', 'SME Owner']
base_rates = [0.82, 0.77, 0.71, 0.79]
for i, (grp, base) in enumerate(zip(groups, base_rates)):
    rates = np.clip(
        base + np.cumsum(np.random.normal(-0.005, 0.02, n_months)), 0.55, 0.95
    )
    colors_fair = [PAL['danger'], PAL['gold'], PAL['success'], PAL['accent']]
    ax.plot(range(n_months), rates, lw=2, marker='o', ms=4,
             label=grp, color=colors_fair[i])

ax.axhline(0.80 * max(base_rates), color=PAL['danger'], lw=2, linestyle='--',
            label='80% parity line')
ax.set_title('Fairness Monitoring: Approval Rate by Segment', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Approval Rate')
ax.set_xticks(range(n_months))
ax.set_xticklabels([f'M{i+1}' for i in range(n_months)], fontsize=8)
ax.legend(fontsize=8)

plt.tight_layout()
savefig('04_mlops_advanced_analytics.png', folder=MLOPS_DIR / 'dashboards')
print('  ✅  Advanced Analytics Dashboard saved.')
log.info('Advanced Visual Analytics complete.')


══════════════════════════════════════════════════════════════════════
  STEP 18 — ADVANCED VISUAL ANALYTICS
══════════════════════════════════════════════════════════════════════
13:54:00 | INFO     |   Figure saved: 04_mlops_advanced_analytics.png
  ✅  Advanced Analytics Dashboard saved.
13:54:00 | INFO     | Advanced Visual Analytics complete.


In [17]:
# =============================================================================
# CELL 16 — STEP 19: MLOps Streamlit Control Center
# =============================================================================
section('STEP 19 — MLOPS STREAMLIT CONTROL CENTER')

MLOPS_STREAMLIT_CODE = '''
"""
╔══════════════════════════════════════════════════════════════════════╗
║  FINLEND AI — MLOPS CONTROL CENTER                                  ║
║  Module 13 — Enterprise MLOps & Model Operations                    ║
║  Run: streamlit run mlops_control_center.py --server.port 8503      ║
╚══════════════════════════════════════════════════════════════════════╝
"""
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime
import json, time

st.set_page_config(
    page_title="FinLend AI — MLOps Control Center",
    page_icon="⚙️",
    layout="wide",
    initial_sidebar_state="expanded",
)

PAL = {
    'navy': '#0A1628', 'accent': '#00D4FF', 'success': '#00C896',
    'danger': '#FF4757', 'warning': '#FFA502', 'gold': '#FFB800',
    'purple': '#7B2FBE', 'neutral': '#8892A4', 'bg': '#F0F4F8',
}

st.markdown(f"""
<style>
  html, body, [class*="css"] {{ font-family: "DM Sans", sans-serif; }}
  .main {{ background: {PAL["bg"]}; }}
  .mlops-header {{
    background: linear-gradient(135deg, {PAL["navy"]} 0%, #1A3A6B 100%);
    color: white; padding: 20px 28px; border-radius: 14px;
    margin-bottom: 22px; border: 1px solid rgba(0,212,255,0.2);
  }}
  .kpi-card {{
    background: white; border-radius: 12px; padding: 18px 20px;
    border-top: 3px solid {PAL["accent"]};
    box-shadow: 0 1px 8px rgba(15,23,42,0.07);
  }}
  .model-card {{
    background: white; border-radius: 10px; padding: 16px 18px;
    margin: 6px 0; border-left: 4px solid;
    box-shadow: 0 1px 6px rgba(0,0,0,0.06);
  }}
  .alert-red    {{ background: #FEF2F2; border-left: 4px solid #EF4444 !important; }}
  .alert-yellow {{ background: #FFFBEB; border-left: 4px solid #F59E0B !important; }}
  .alert-green  {{ background: #ECFDF5; border-left: 4px solid #10B981 !important; }}
</style>
""", unsafe_allow_html=True)

# ── Header ────────────────────────────────────────────────────────────────────
st.markdown(f"""
<div class="mlops-header">
  <div style="font-size:24px;font-weight:700;color:{PAL["accent"]};
              font-family:monospace;">⚙️ FinLend AI — MLOps Control Center</div>
  <div style="font-size:13px;color:rgba(255,255,255,0.55);margin-top:4px;">
    Enterprise Model Operations · Drift Monitoring · Governance · Retraining
    · {datetime.now().strftime("%B %d, %Y  %H:%M")}
  </div>
</div>
""", unsafe_allow_html=True)

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown(f"""
    <div style="background:{PAL["navy"]};padding:16px;border-radius:10px;margin-bottom:12px;">
      <div style="color:{PAL["accent"]};font-size:16px;font-weight:700;">⚙️ MLOps Platform</div>
      <div style="color:{PAL["neutral"]};font-size:11px;">Module 13 — Control Center</div>
    </div>
    """, unsafe_allow_html=True)

    page = st.radio("Navigation", [
        "🏠 Platform Overview",
        "📊 Model Registry",
        "🔄 Drift Monitor",
        "🔁 Retraining",
        "⚖️ Governance",
        "🔌 API Health",
        "🚨 Alerts",
    ])


# ── Data Simulation ────────────────────────────────────────────────────────────
@st.cache_data(ttl=60)
def generate_platform_data():
    np.random.seed(42)
    n = 168
    ts = pd.date_range("2024-12-01", periods=n, freq="h")
    return pd.DataFrame({
        "timestamp":      ts,
        "requests":       np.random.poisson(450, n),
        "latency_p99":    np.clip(180 + np.random.normal(0, 30, n), 50, 600),
        "error_rate":     np.clip(np.random.exponential(0.3, n), 0, 5),
        "pd_score_mean":  np.clip(0.18 + np.cumsum(np.random.normal(0, 0.001, n)), 0.05, 0.55),
        "fraud_alerts":   np.random.poisson(8, n) + (np.arange(n) > 120) * 5,
        "model_uptime":   np.clip(99.9 - np.random.exponential(0.05, n), 98.0, 100.0),
    })


df_mon = generate_platform_data()

MODELS = {
    "PD_Model":    {"version": "v2.1", "stage": "Production", "auc": 0.812, "psi": 0.089, "last_retrained": "2024-11-28"},
    "Fraud_Model": {"version": "v1.3", "stage": "Production", "auc": 0.921, "psi": 0.142, "last_retrained": "2024-11-22"},
    "EWS_Model":   {"version": "v1.1", "stage": "Staging",    "auc": 0.774, "psi": 0.063, "last_retrained": "2024-12-01"},
}


# ── Page Routing ──────────────────────────────────────────────────────────────
if "Overview" in page:
    c1, c2, c3, c4, c5 = st.columns(5)
    with c1:
        st.markdown(f"""
        <div class="kpi-card">
          <div style="font-size:11px;color:{PAL["neutral"]}">Models in Production</div>
          <div style="font-size:28px;font-weight:700;">2</div>
        </div>""", unsafe_allow_html=True)
    with c2:
        avg_p99 = df_mon["latency_p99"].mean()
        color = PAL["danger"] if avg_p99 > 200 else PAL["success"]
        st.markdown(f"""
        <div class="kpi-card" style="border-top-color:{color};">
          <div style="font-size:11px;color:{PAL["neutral"]}">Avg P99 Latency</div>
          <div style="font-size:28px;font-weight:700;">{avg_p99:.0f}ms</div>
        </div>""", unsafe_allow_html=True)
    with c3:
        uptime = df_mon["model_uptime"].mean()
        color = PAL["success"] if uptime >= 99.9 else PAL["warning"]
        st.markdown(f"""
        <div class="kpi-card" style="border-top-color:{color};">
          <div style="font-size:11px;color:{PAL["neutral"]}">Model Uptime</div>
          <div style="font-size:28px;font-weight:700;">{uptime:.2f}%</div>
        </div>""", unsafe_allow_html=True)
    with c4:
        max_psi = max(m["psi"] for m in MODELS.values())
        color = PAL["danger"] if max_psi > 0.20 else PAL["warning"] if max_psi > 0.10 else PAL["success"]
        st.markdown(f"""
        <div class="kpi-card" style="border-top-color:{color};">
          <div style="font-size:11px;color:{PAL["neutral"]}">Max PSI (Drift)</div>
          <div style="font-size:28px;font-weight:700;">{max_psi:.3f}</div>
        </div>""", unsafe_allow_html=True)
    with c5:
        total_reqs = df_mon["requests"].sum()
        st.markdown(f"""
        <div class="kpi-card">
          <div style="font-size:11px;color:{PAL["neutral"]}">Requests (7d)</div>
          <div style="font-size:28px;font-weight:700;">{total_reqs:,}</div>
        </div>""", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    c1, c2 = st.columns(2)
    with c1:
        st.subheader("API Request Volume (7 days)")
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=df_mon["timestamp"], y=df_mon["requests"],
            fill="tozeroy", line=dict(color=PAL["accent"], width=1.5),
            fillcolor="rgba(0,212,255,0.15)",
        ))
        fig.update_layout(height=300, paper_bgcolor="white",
                          yaxis_title="Requests/hr",
                          margin=dict(l=0, r=0, t=10, b=0))
        st.plotly_chart(fig, use_container_width=True)
    with c2:
        st.subheader("P99 Latency Trend")
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=df_mon["timestamp"], y=df_mon["latency_p99"],
            line=dict(color=PAL["warning"], width=1.5),
        ))
        fig.add_hline(y=200, line_dash="dash", line_color=PAL["danger"],
                       annotation_text="SLO (200ms)")
        fig.update_layout(height=300, paper_bgcolor="white",
                          yaxis_title="Latency (ms)",
                          margin=dict(l=0, r=0, t=10, b=0))
        st.plotly_chart(fig, use_container_width=True)


elif "Registry" in page:
    st.subheader("📊 Model Registry — Registered Models")
    for model_name, info in MODELS.items():
        psi_color = PAL["danger"] if info["psi"] > 0.20 else PAL["warning"] if info["psi"] > 0.10 else PAL["success"]
        stage_color = PAL["success"] if info["stage"] == "Production" else PAL["warning"]
        st.markdown(f"""
        <div class="model-card" style="border-color:{stage_color};">
          <div style="display:flex;justify-content:space-between;align-items:center;">
            <div>
              <span style="font-weight:700;font-size:15px;">{model_name}</span>
              <span style="background:{stage_color};color:white;padding:3px 10px;
                border-radius:12px;font-size:11px;margin-left:10px;">{info["stage"]}</span>
            </div>
            <span style="font-size:12px;color:{PAL["neutral"]}">Version: {info["version"]}</span>
          </div>
          <div style="margin-top:10px;display:flex;gap:20px;font-size:13px;">
            <span>🎯 AUC-ROC: <b>{info["auc"]:.3f}</b></span>
            <span style="color:{psi_color};">📊 PSI: <b>{info["psi"]:.3f}</b></span>
            <span>📅 Retrained: {info["last_retrained"]}</span>
          </div>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    st.subheader("Model Governance Actions")
    c1, c2, c3 = st.columns(3)
    with c1:
        model_sel = st.selectbox("Select Model", list(MODELS.keys()))
    with c2:
        action_sel = st.selectbox("Action", ["Promote to Production", "Move to Staging", "Archive", "Rollback"])
    with c3:
        approver_name = st.text_input("Approver Name", "CRO")
    if st.button("✅ Execute Governance Action", type="primary"):
        st.success(f"✅ Action '{action_sel}' applied to {model_sel} by {approver_name} at {datetime.now().strftime('%H:%M:%S')}")


elif "Drift" in page:
    st.subheader("🔄 Feature Drift Monitor")
    np.random.seed(42)
    features = ["credit_score", "pd_score", "financial_stress_index",
                 "missed_payment_ratio", "spending_volatility", "debt_to_income",
                 "income_stability", "loan_amount"]
    psi_vals = np.array([0.18, 0.14, 0.22, 0.09, 0.07, 0.19, 0.05, 0.03])
    ks_vals  = np.array([0.11, 0.09, 0.13, 0.06, 0.04, 0.12, 0.03, 0.02])

    drift_data = pd.DataFrame({"Feature": features, "PSI": psi_vals, "KS": ks_vals})
    drift_data["Status"] = drift_data["PSI"].apply(
        lambda x: "RETRAIN" if x >= 0.25 else "ALERT" if x >= 0.20 else "MONITOR" if x >= 0.10 else "OK"
    )
    drift_data["Color"] = drift_data["Status"].map({
        "RETRAIN": PAL["danger"], "ALERT": PAL["warning"],
        "MONITOR": PAL["gold"], "OK": PAL["success"]
    })

    fig = go.Figure(go.Bar(
        x=drift_data["PSI"], y=drift_data["Feature"],
        orientation="h",
        marker_color=drift_data["Color"].tolist(),
        text=[f"{v:.3f} ({s})" for v, s in zip(drift_data["PSI"], drift_data["Status"])],
        textposition="outside",
    ))
    fig.add_vline(x=0.10, line_dash="dash", line_color=PAL["gold"],    annotation_text="Monitor")
    fig.add_vline(x=0.20, line_dash="dash", line_color=PAL["warning"], annotation_text="Alert")
    fig.add_vline(x=0.25, line_dash="dash", line_color=PAL["danger"],  annotation_text="Retrain")
    fig.update_layout(height=400, paper_bgcolor="white",
                      xaxis_title="PSI Score",
                      margin=dict(l=0, r=80, t=20, b=0))
    st.plotly_chart(fig, use_container_width=True)

    n_alert = (drift_data["Status"].isin(["ALERT", "RETRAIN"])).sum()
    if n_alert > 0:
        st.warning(f"⚠️ {n_alert} feature(s) require attention. Consider triggering retraining.")
    else:
        st.success("✅ All features within acceptable drift thresholds.")


elif "Retraining" in page:
    st.subheader("🔁 Retraining Control Panel")
    c1, c2, c3 = st.columns(3)
    with c1:
        retrain_model = st.selectbox("Model", list(MODELS.keys()))
    with c2:
        trigger_type = st.selectbox("Trigger Reason",
            ["Manual", "PSI Drift", "AUC Degradation", "Fairness Drift", "Scheduled"])
    with c3:
        st.markdown("<br>", unsafe_allow_html=True)
        launch = st.button("🚀 Launch Retraining", type="primary")

    if launch:
        with st.spinner(f"⚙️ Running retraining pipeline for {retrain_model}..."):
            import time as t_lib
            t_lib.sleep(1.5)
        before_auc = np.random.uniform(0.72, 0.78)
        after_auc  = np.random.uniform(0.79, 0.87)
        st.success(f"""
        ✅ Retraining complete!
        Model: {retrain_model} | Trigger: {trigger_type}
        AUC: {before_auc:.3f} → {after_auc:.3f} (+{(after_auc-before_auc):.3f})
        Status: GOVERNANCE GATE PASSED — Ready for staging review.
        """)

    st.subheader("Recent Retraining Jobs")
    jobs_data = pd.DataFrame([
        {"Job ID": "RTJ-A1B2C3", "Model": "PD_Model",    "Trigger": "PSI Alert",    "Status": "✅ PASSED", "AUC Δ": "+0.031", "Date": "2024-11-28"},
        {"Job ID": "RTJ-D4E5F6", "Model": "Fraud_Model", "Trigger": "Monthly",      "Status": "❌ FAILED", "AUC Δ": "-0.012", "Date": "2024-11-15"},
        {"Job ID": "RTJ-G7H8I9", "Model": "Fraud_Model", "Trigger": "Retry",        "Status": "✅ PASSED", "AUC Δ": "+0.018", "Date": "2024-11-22"},
        {"Job ID": "RTJ-J0K1L2", "Model": "EWS_Model",   "Trigger": "AUC Degrad.", "Status": "✅ PASSED", "AUC Δ": "+0.028", "Date": "2024-12-01"},
    ])
    st.dataframe(jobs_data, use_container_width=True, hide_index=True)


elif "Governance" in page:
    st.subheader("⚖️ AI Governance & Compliance")
    gov_checks = [
        ("Model Explainability",    True,  "SHAP values logged for 100% of decisions"),
        ("Audit Trail Active",      True,  "All predictions persisted to audit_log"),
        ("Fairness Monitoring",     True,  "DI ratio > 0.80 across all groups"),
        ("Model Approval Workflow", True,  "CRO approval required before production"),
        ("PSI Monitoring",          True,  "Drift monitored weekly; alert at PSI>0.20"),
        ("Rollback Capability",     True,  "All model versions retained in registry"),
        ("RBI Compliance",          True,  "Fair Practices Code implemented"),
        ("PMLA Fraud Logging",      True,  "All fraud alerts logged with SLA tracking"),
    ]
    cols = st.columns(2)
    for i, (check, ok, detail) in enumerate(gov_checks):
        with cols[i % 2]:
            if ok:
                st.success(f"✅ **{check}**\\n\\n{detail}")
            else:
                st.error(f"❌ **{check}**\\n\\n{detail}")


elif "API" in page:
    st.subheader("🔌 API Health & Performance")
    endpoints = [
        {"Endpoint": "/v1/underwriting/score", "Status": "🟢 Healthy", "Avg Latency": "48ms",  "P99": "182ms", "RPS": "142"},
        {"Endpoint": "/v1/fraud/score",        "Status": "🟢 Healthy", "Avg Latency": "18ms",  "P99": "67ms",  "RPS": "89"},
        {"Endpoint": "/v1/ews/score",           "Status": "🟢 Healthy", "Avg Latency": "52ms",  "P99": "198ms", "RPS": "34"},
        {"Endpoint": "/health",                 "Status": "🟢 Healthy", "Avg Latency": "2ms",   "P99": "8ms",   "RPS": "15"},
        {"Endpoint": "/v1/metrics/predictions", "Status": "🟢 Healthy", "Avg Latency": "12ms",  "P99": "45ms",  "RPS": "5"},
    ]
    st.dataframe(pd.DataFrame(endpoints), use_container_width=True, hide_index=True)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_mon["timestamp"], y=df_mon["model_uptime"],
        fill="tozeroy", fillcolor="rgba(0,200,150,0.1)",
        line=dict(color=PAL["success"], width=2),
        name="Model Uptime",
    ))
    fig.add_hline(y=99.9, line_dash="dash", line_color=PAL["danger"],
                   annotation_text="SLO (99.9%)")
    fig.update_layout(height=300, paper_bgcolor="white",
                      yaxis_title="Uptime (%)", yaxis_range=[98, 100.2],
                      margin=dict(l=0, r=0, t=10, b=0))
    st.plotly_chart(fig, use_container_width=True)


elif "Alerts" in page:
    st.subheader("🚨 Active Governance & Operational Alerts")
    alerts = [
        {"⏰": "2 min ago",  "Severity": "🔴 CRITICAL", "Source": "Drift Detector",    "Message": "financial_stress_index PSI=0.22 — Alert threshold exceeded",   "Action": "Trigger retraining within 24h"},
        {"⏰": "15 min ago", "Severity": "🟠 HIGH",     "Source": "Governance Engine", "Message": "Fraud_Model PSI=0.142 approaching alert threshold (0.15)",        "Action": "Increase monitoring to hourly"},
        {"⏰": "1 hr ago",   "Severity": "🟠 HIGH",     "Source": "API Monitor",       "Message": "Underwriting API P99 latency=218ms — SLO breach (>200ms)",      "Action": "Scale up API pods or optimize query"},
        {"⏰": "3 hr ago",   "Severity": "🟡 MEDIUM",   "Source": "EWS Model",         "Message": "EWS recall dropped to 0.61 (threshold: 0.65)",                  "Action": "Review model performance; retrain if persistent"},
        {"⏰": "12 hr ago",  "Severity": "🟢 INFO",     "Source": "Retraining",        "Message": "PD_Model v2.1 retrained and promoted to production successfully", "Action": "No action required"},
    ]
    for alert in alerts:
        color = "alert-red" if "CRITICAL" in alert["Severity"] else \\
                "alert-yellow" if "HIGH" in alert["Severity"] else \\
                "alert-yellow" if "MEDIUM" in alert["Severity"] else "alert-green"
        st.markdown(f"""
        <div class="model-card {color}">
          <div style="display:flex;justify-content:space-between;">
            <span style="font-weight:700;">{alert["Severity"]} — {alert["Source"]}</span>
            <span style="color:#94A3B8;font-size:12px;">{alert["⏰"]}</span>
          </div>
          <div style="font-size:13px;margin-top:6px;">{alert["Message"]}</div>
          <div style="font-size:12px;color:#4A5568;margin-top:4px;">→ {alert["Action"]}</div>
        </div>
        """, unsafe_allow_html=True)

# Footer
st.markdown(f"""
<div style="margin-top:32px;padding:12px;background:white;border-radius:10px;
            border-top:2px solid {PAL["accent"]}40;text-align:center;">
  <span style="color:{PAL["neutral"]};font-size:11px;">
    FinLend AI MLOps Control Center · Module 13 · {datetime.now().strftime("%H:%M:%S")}
  </span>
</div>
""", unsafe_allow_html=True)
'''

streamlit_path = MLOPS_DIR / 'mlops_control_center.py'
with open(streamlit_path, 'w', encoding='utf-8') as f:
    f.write(MLOPS_STREAMLIT_CODE)

print(f'  ✅  MLOps Streamlit Control Center saved: {streamlit_path}')
log.info('MLOps Streamlit Control Center complete.')


══════════════════════════════════════════════════════════════════════
  STEP 19 — MLOPS STREAMLIT CONTROL CENTER
══════════════════════════════════════════════════════════════════════
  ✅  MLOps Streamlit Control Center saved: C:\Users\abhir\OneDrive\Desktop\iitg\mlops_platform\mlops_control_center.py
13:54:01 | INFO     | MLOps Streamlit Control Center complete.


In [18]:
# =============================================================================
# CELL 17 — Test Suite
# =============================================================================
section('TEST SUITE — API & PIPELINE TESTS')

TEST_CODE = '''
"""
FinLend AI MLOps — Test Suite
Unit tests for drift detection, feature store, and API.
"""
import pytest
import numpy as np
import pandas as pd
import sys, os

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)) + '/..')


# ── Drift Detector Tests ──────────────────────────────────────────────────────
class TestDriftDetector:

    def setup_method(self):
        """Import DriftDetector from module (adjust path as needed)."""
        # Inline implementation for testing
        import numpy as np
        from scipy import stats

        def compute_psi(baseline, current, n_bins=10):
            eps = 1e-6
            bins = np.percentile(baseline, np.linspace(0, 100, n_bins + 1))
            bins = np.unique(bins)
            if len(bins) < 3:
                return 0.0
            bp = np.maximum(np.histogram(baseline, bins=bins)[0] / len(baseline), eps)
            cp = np.maximum(np.histogram(current,  bins=bins)[0] / len(current),  eps)
            bp /= bp.sum(); cp /= cp.sum()
            return float(np.sum((cp - bp) * np.log(cp / bp + eps)))

        self.compute_psi = compute_psi

    def test_psi_stable_distribution(self):
        """PSI should be near zero for identical distributions."""
        np.random.seed(42)
        base = np.random.normal(0.2, 0.05, 1000)
        curr = np.random.normal(0.2, 0.05, 1000)
        psi = self.compute_psi(base, curr)
        assert psi < 0.10, f"Stable distribution PSI={psi:.3f} should be < 0.10"

    def test_psi_drifted_distribution(self):
        """PSI should be high for significantly shifted distributions."""
        np.random.seed(42)
        base = np.random.normal(0.20, 0.05, 1000)
        curr = np.random.normal(0.45, 0.08, 1000)  # Major shift
        psi = self.compute_psi(base, curr)
        assert psi > 0.20, f"Drifted distribution PSI={psi:.3f} should be > 0.20"

    def test_psi_is_non_negative(self):
        np.random.seed(99)
        base = np.random.beta(2, 5, 500)
        curr = np.random.beta(3, 4, 500)
        psi = self.compute_psi(base, curr)
        assert psi >= 0, f"PSI must be non-negative, got {psi}"


# ── PD Score Logic Tests ──────────────────────────────────────────────────────
class TestPDScoringLogic:

    def compute_pd(self, credit_score=680, missed_payment_ratio=0.05,
                    worst_delinquency_stage=0, financial_stress_index=0.3,
                    avg_delay_days=5, debt_to_income_ratio=2.0,
                    spending_volatility_index=0.25, credit_history_length=36):
        cs_norm   = max(0, (900 - credit_score) / 600)
        mpr_norm  = min(missed_payment_ratio, 1.0)
        dpd_norm  = worst_delinquency_stage / 4.0
        dly_norm  = min(avg_delay_days / 90, 1.0)
        fsi_norm  = min(financial_stress_index, 1.0)
        dti_norm  = min(debt_to_income_ratio / 10, 1.0)
        svi_norm  = min(spending_volatility_index, 1.0)
        hist_norm = max(0, 1 - credit_history_length / 60)
        raw_pd = (
            0.22 * mpr_norm + 0.20 * dpd_norm + 0.12 * cs_norm
            + 0.15 * fsi_norm + 0.10 * dly_norm + 0.08 * dti_norm
            + 0.08 * svi_norm + 0.05 * hist_norm
        )
        if credit_history_length < 6:
            raw_pd = min(raw_pd + 0.06, 0.95)
        if worst_delinquency_stage >= 3:
            raw_pd = min(raw_pd + 0.15, 0.95)
        return float(np.clip(raw_pd, 0.01, 0.95))

    def test_prime_borrower_low_pd(self):
        """Prime borrowers (high score, no delinquency) should have low PD."""
        pd = self.compute_pd(credit_score=850, missed_payment_ratio=0.0,
                              worst_delinquency_stage=0, financial_stress_index=0.1,
                              credit_history_length=60)
        assert pd < 0.15, f"Prime borrower PD={pd:.3f} should be < 0.15"

    def test_risky_borrower_high_pd(self):
        """High-risk borrowers should have high PD."""
        pd = self.compute_pd(credit_score=350, missed_payment_ratio=0.6,
                              worst_delinquency_stage=3, financial_stress_index=0.8,
                              debt_to_income_ratio=8, credit_history_length=2)
        assert pd > 0.55, f"Risky borrower PD={pd:.3f} should be > 0.55"

    def test_pd_bounded(self):
        """PD must be in [0.01, 0.95]."""
        pd1 = self.compute_pd(credit_score=900, missed_payment_ratio=0.0)
        pd2 = self.compute_pd(credit_score=300, missed_payment_ratio=1.0,
                               worst_delinquency_stage=4)
        assert 0.01 <= pd1 <= 0.95, f"PD1={pd1} out of bounds"
        assert 0.01 <= pd2 <= 0.95, f"PD2={pd2} out of bounds"

    def test_delinquency_bump(self):
        """DPD stage 3+ should increase PD."""
        pd_clean = self.compute_pd(worst_delinquency_stage=0)
        pd_dpd3  = self.compute_pd(worst_delinquency_stage=3)
        assert pd_dpd3 > pd_clean, f"DPD3 PD should be > clean PD: {pd_dpd3:.3f} vs {pd_clean:.3f}"


# ── Fraud Scoring Tests ───────────────────────────────────────────────────────
class TestFraudScoring:

    def compute_fraud(self, monthly_income=55000, employment_type='Salaried',
                       credit_history_length=36, loan_amount=200000,
                       worst_delinquency_stage=0, income_stability_score=0.7):
        medians = {'Salaried': 45000, 'Self-Employed': 52000,
                    'Gig Worker': 22000, 'SME Owner': 85000, 'First-Time Borrower': 18000}
        exp_income = medians.get(employment_type, 40000)
        inflation = min(max((monthly_income / max(exp_income, 1)) - 1.0, 0) / 2.0, 1.0)
        doc_risk = float(
            (credit_history_length < 6) * 0.30
            + (worst_delinquency_stage >= 3) * 0.20
            + (1 - income_stability_score) * 0.25
        )
        identity = 1 - min(
            0.90 + (income_stability_score - 0.5) * 0.20
            - (credit_history_length < 6) * 0.20, 1.0
        )
        synthetic = float(
            (credit_history_length < 6) * 0.40
            + (loan_amount > monthly_income * 20) * 0.25
            + doc_risk * 0.20
        )
        return float(np.clip(
            0.35 * inflation + 0.25 * doc_risk + 0.25 * identity + 0.15 * synthetic, 0, 1
        ))

    def test_low_fraud_normal_applicant(self):
        score = self.compute_fraud()
        assert score < 0.35, f"Normal applicant fraud score={score:.3f} should be < 0.35"

    def test_high_fraud_thin_file(self):
        score = self.compute_fraud(credit_history_length=2, income_stability_score=0.2,
                                    worst_delinquency_stage=3)
        assert score > 0.35, f"Thin-file fraud score={score:.3f} should be > 0.35"

    def test_income_inflation_raises_score(self):
        normal_score = self.compute_fraud(monthly_income=45000)
        inflated_score = self.compute_fraud(monthly_income=150000)
        assert inflated_score > normal_score, "Inflated income should raise fraud score"

    def test_fraud_score_bounded(self):
        score = self.compute_fraud(credit_history_length=0, monthly_income=999999,
                                    loan_amount=10000000)
        assert 0 <= score <= 1, f"Fraud score {score} out of [0,1] bounds"


if __name__ == '__main__':
    pytest.main(['--tb=short', '-v', __file__])
'''

test_path = MLOPS_DIR / 'tests' / 'test_mlops_core.py'
with open(test_path, 'w', encoding='utf-8') as f:
    f.write(TEST_CODE)

print(f'  ✅  Test suite saved: {test_path}')
log.info('Test suite complete.')


══════════════════════════════════════════════════════════════════════
  TEST SUITE — API & PIPELINE TESTS
══════════════════════════════════════════════════════════════════════
  ✅  Test suite saved: C:\Users\abhir\OneDrive\Desktop\iitg\mlops_platform\tests\test_mlops_core.py
13:54:06 | INFO     | Test suite complete.


In [19]:
# =============================================================================
# CELL 18 — STEP 20: Production Runbook & Documentation
# =============================================================================
section('STEP 20 — PRODUCTION RUNBOOK & FINAL DOCUMENTATION')

PRODUCTION_RUNBOOK = """
================================================================================
  FINLEND AI — MLOPS PRODUCTION RUNBOOK
  Module 13 — Enterprise MLOps & Production Operations
  Version: 1.0.0
================================================================================

PLATFORM QUICK START
────────────────────
1. Start all services:
   cd iitg/mlops_platform
   docker-compose -f docker/docker-compose.yml up -d

2. Access points:
   Model API:     http://localhost:8000/docs
   MLflow UI:     http://localhost:5000
   Control Center: streamlit run mlops_control_center.py
   Lending Platform: streamlit run app.py --server.port 8502

3. Run tests:
   pytest mlops_platform/tests/ -v

OPERATIONAL PLAYBOOKS
─────────────────────

Playbook 1: Model Drift Alert
  TRIGGER: PSI > 0.20 on any feature
  STEPS:
    1. MLOps Control Center → Drift Monitor → identify drifted features
    2. Notify ML team via Slack/email
    3. Evaluate if drift is temporary (market shock) or structural
    4. If structural: trigger retraining (Retraining tab → Launch Retraining)
    5. Monitor retrained model in staging for 48h
    6. CRO approval → promote to production

Playbook 2: Latency SLO Breach
  TRIGGER: P99 > 200ms
  STEPS:
    1. Inspect API Logs (logs/module13.log)
    2. Check container resource utilization (docker stats)
    3. Scale up replicas if under high load: docker-compose up --scale finlend-api=4
    4. If issue persists, check for slow SQL queries in Feature Store

Playbook 3: Emergency Rollback
  TRIGGER: Sudden model degradation or error spike
  STEPS:
    1. Open MLOps Control Center → Model Registry
    2. Select production model → Click "Rollback" to previous stable version
    3. Verify API health check status
    4. Notify stakeholders & initiate post-mortem
"""
print(PRODUCTION_RUNBOOK)
with open(MLOPS_DIR / 'docs' / 'production_runbook.txt', 'w', encoding='utf-8') as f:
    f.write(PRODUCTION_RUNBOOK)
log.info('Production Runbook saved.')


══════════════════════════════════════════════════════════════════════
  STEP 20 — PRODUCTION RUNBOOK & FINAL DOCUMENTATION
══════════════════════════════════════════════════════════════════════

  FINLEND AI — MLOPS PRODUCTION RUNBOOK
  Module 13 — Enterprise MLOps & Production Operations
  Version: 1.0.0

PLATFORM QUICK START
────────────────────
1. Start all services:
   cd iitg/mlops_platform
   docker-compose -f docker/docker-compose.yml up -d

2. Access points:
   Model API:     http://localhost:8000/docs
   MLflow UI:     http://localhost:5000
   Control Center: streamlit run mlops_control_center.py
   Lending Platform: streamlit run app.py --server.port 8502

3. Run tests:
   pytest mlops_platform/tests/ -v

OPERATIONAL PLAYBOOKS
─────────────────────

Playbook 1: Model Drift Alert
  TRIGGER: PSI > 0.20 on any feature
  STEPS:
    1. MLOps Control Center → Drift Monitor → identify drifted features
    2. Notify ML team via Slack/email
    3. Evaluate if drift is temporary (mar